# FPL VORTEX · xG MIX + DEFCON + GAME STATE

One full 20-team chart + matching X caption per report:

- `MyDrive/FPL_xG_Mix/xG/1.png` and `1.txt`
- `MyDrive/FPL_xG_Mix/DEFCON/1.png` and `1.txt`
- `MyDrive/FPL_xG_Mix/GAME_STATE/1.png` and `1.txt`

**GAME_STATE source:** FotMob goal events, verified against FPL final scores. Its normalized verified goal timelines are **persistently archived** under `_assets/gamestate_match_cache/verified_fotmob_events/`. Verified previous matches do not require future FotMob/GitHub availability. New matches still require an accessible authorized source. No invented times or partial-season publications.


In [1]:
%pip install -q numpy pandas requests matplotlib pillow "playwright==1.55.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 13.8 MB/s eta 0:00:00


## 1 · Setup, API/data, visual design & X caption system

The branding constants **X_HANDLE**, **YOUTUBE_HANDLE**, and **DEFAULT_LOGO** can be changed here. The image loader preserves image caching and handles unavailable PL headshots.

In [2]:
import argparse
import datetime as dt
import difflib
import io
import json
import random
import re
import warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.font_manager import FontProperties
from matplotlib.patches import Rectangle, FancyBboxPatch, Polygon
from PIL import Image

FPL_URL = "https://fantasy.premierleague.com/api/bootstrap-static/"
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; fpl-vortex-xg/3.0)"}
DRIVE_FOLDER = "/content/drive/MyDrive/FPL_xG_Mix"
CHANNEL = "FPL VORTEX"
X_HANDLE = "@FPLVORTEXM"
YOUTUBE_HANDLE = "@FPLVORTEX"
BADGE_URLS = [  # tried in order; letter badge is used if all fail
    "https://resources.premierleague.com/premierleague/badges/70/t{code}.png",
    "https://resources.premierleague.com/premierleague/badges/t{code}.png",
    "https://resources.premierleague.com/premierleague25/badges/70/t{code}.png",
]

# (column, label, colour) - stacking order left to right
CATS_SITUATION = [
    ("regular_play", "Regular Play", "#27AE60"),
    ("free_kicks", "Free Kicks", "#8E68D6"),
    ("penalties", "Penalties", "#E56B7B"),
    ("corners", "Corners", "#3B82F6"),
    ("throw_ins", "Throw-ins", "#45B5AA"),
    ("fast_breaks", "Fast Breaks", "#F28C38"),
    ("other", "Other", "#BDBDBD"),
]
CATS_POSITION = [
    ("forwards", "Forwards", "#27AE60"),
    ("midfielders", "Midfielders", "#3B82F6"),
    ("defenders", "Defenders", "#F28C38"),
    ("goalkeepers", "Goalkeepers", "#8E68D6"),
]

UNDERSTAT_MAP = {
    "OpenPlay": "regular_play",
    "DirectFreekick": "free_kicks",
    "Penalty": "penalties",
    "FromCorner": "corners",
    "SetPiece": "other",
}
ALIASES = {  # Understat name -> FPL name (the rest is fuzzy-matched)
    "Manchester City": "Man City",
    "Manchester United": "Man Utd",
    "Newcastle United": "Newcastle",
    "Wolverhampton Wanderers": "Wolves",
    "Tottenham": "Spurs",
    "Nottingham Forest": "Nott'm Forest",
}

BG, INK, GRID = "#F4EFE4", "#111827", "#D9D2C3"
BRAND_BG, GOOD, BAD = "#0B1B3A", "#1E8E4E", "#C0392B"
MIN_LABEL_PCT = 4.0


FONT_BASE = "https://raw.githubusercontent.com/google/fonts/main/ofl/"
FONT_FILES = {  # Bebas Neue for the title, Poppins for everything else
    "title": "bebasneue/BebasNeue-Regular.ttf",
    "reg": "poppins/Poppins-Regular.ttf",
    "semi": "poppins/Poppins-SemiBold.ttf",
    "bold": "poppins/Poppins-Bold.ttf",
}
FONTS = {}
DEFAULT_LOGO = "https://drive.google.com/file/d/17RWQXfSOrrCpMt9JbXz6z1J94JSaPQ2Z/view?usp=drivesdk"


def load_fonts(font_dir):
    """Download (once) and load the fonts. Falls back to Matplotlib's default."""
    FONTS.clear()
    d = Path(font_dir)
    d.mkdir(parents=True, exist_ok=True)
    for key, rel in FONT_FILES.items():
        f = d / Path(rel).name
        if not f.exists():
            try:
                r = requests.get(FONT_BASE + rel, headers=HEADERS, timeout=30)
                r.raise_for_status()
                if r.content[:4] not in (b"\x00\x01\x00\x00", b"OTTO", b"true"):
                    raise ValueError("not a font file")
                f.write_bytes(r.content)
            except Exception as e:
                print(f"Font {f.name} unavailable ({e}) - using default font.")
                continue
        try:
            FONTS[key] = FontProperties(fname=str(f))
        except Exception:
            pass


def T(ax, x, y, s, **kw):
    """ax.text with the premium fonts. font= title|reg|semi|bold."""
    key = kw.pop("font", None)
    weight = kw.pop("fontweight", None)
    kw.pop("family", None)
    kw.pop("style", None)
    key = key or ("bold" if weight == "bold" else "reg")
    if key in FONTS:
        kw["fontproperties"] = FONTS[key]
    elif weight:
        kw["fontweight"] = weight
    return ax.text(x, y, s, **kw)


def resolve_logo(arg, cache_dir):
    """Local path, or a Google Drive link / file id. Cached after first download."""
    if not arg or str(arg).lower() == "none":
        return None
    if Path(str(arg)).exists():
        return Path(str(arg))
    m = re.search(r"/d/([A-Za-z0-9_-]+)", str(arg)) or re.search(r"id=([A-Za-z0-9_-]+)", str(arg))
    file_id = m.group(1) if m else str(arg)
    cache = Path(cache_dir) / ("brand_logo_" + re.sub(r"[^A-Za-z0-9]", "_", file_id)[-30:] + ".png")
    if cache.exists():
        return cache
    data = None
    try:  # works if the file is shared as "anyone with the link"
        r = requests.get("https://drive.google.com/uc", headers=HEADERS, timeout=30,
                         params={"export": "download", "id": file_id})
        if r.ok and r.headers.get("content-type", "").startswith("image"):
            data = r.content
    except requests.RequestException:
        pass
    if data is None and in_colab():  # private file: use your own Google login
        try:
            from google.colab import auth
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            auth.authenticate_user()
            svc = build("drive", "v3", cache_discovery=False)
            buf = io.BytesIO()
            dl = MediaIoBaseDownload(buf, svc.files().get_media(fileId=file_id))
            done = False
            while not done:
                _, done = dl.next_chunk()
            data = buf.getvalue()
        except Exception as e:
            print(f"Could not download logo from Drive: {e}")
    if data is None:
        print("Logo not downloaded - drawing the channel-name box instead.")
        return None
    try:
        Image.open(io.BytesIO(data)).convert("RGBA").save(cache)
        return cache
    except Exception:
        print("Downloaded logo file is not a readable image.")
        return None


def clean_old(out_dir, prefix, keep):
    """Delete older charts (and old gameweek markers) in the output folder."""
    for f in Path(out_dir).glob(f"{prefix}*.png"):
        if f != keep:
            f.unlink()
            print(f"Deleted old chart {f.name}")
    for f in Path(out_dir).glob(f"{prefix}*.gw"):
        f.unlink()


# ------------------------------------------------------------- X post text --
X_LIMIT = 280  # Standard X post length target


def x_len(text):
    """Approximate X's weighted length: emoji and symbols count as 2."""
    return sum(2 if ord(c) >= 0x2600 else 1 for c in text)


def _post_data(shares, gw, channel):
    d = {"gw": gw, "channel": channel, "tag": "#" + re.sub(r"\W+", "", channel.title())}
    if {"xg_total", "goals", "diff", "top_share"}.issubset(shares.columns):
        lead = shares.nlargest(1, "xg_total").iloc[0]
        over = shares.nlargest(1, "diff").iloc[0]
        under = shares.nsmallest(1, "diff").iloc[0]
        dep = shares.nlargest(1, "top_share").iloc[0]
        d.update(full=True, lead=lead.team, lead_xg=f"{lead.xg_total:.1f}",
                 over=over.team, over_d=f"{over['diff']:+.1f}",
                 over_g=int(round(over.goals)), over_x=f"{over.xg_total:.1f}",
                 under=under.team, under_d=f"{under['diff']:+.1f}",
                 under_g=int(round(under.goals)), under_x=f"{under.xg_total:.1f}",
                 dep=dep.team, dep_pl=dep.top_player, dep_sh=f"{dep.top_share:.0f}")
    else:
        first = [c for c in shares.columns if c not in ("team",)][0]
        top = shares.nlargest(1, first).iloc[0]
        d.update(full=False, top=top.team, col=first.replace("_", " "),
                 sh=f"{top[first]:.0f}")
    return d


def _templates(d):
    """Each template = (lines, hashtags). Lines are ordered by importance."""
    g = d["gw"]
    if not d["full"]:
        return [([f"\U0001F4CA GW{g} xG mix | {d['channel']}",
                  f"\U0001F525 {d['top']} lead on {d['col']}: {d['sh']}% of their xG"],
                 f"#FPL #xG #GW{g} #FPLVORTEX")]
    return [
        ([f"\U0001F4CA GW{g} xG Mix | {d['channel']}",
          f"\U0001F525 {d['lead']} lead the league: {d['lead_xg']} xG",
          f"\U0001F3AF {d['over']} {d['over_d']} vs xG ({d['over_g']} goals, {d['over_x']} xG)",
          f"\U0001F9CA {d['under']} {d['under_d']} vs xG",
          f"\u26A0\uFE0F {d['dep']} lean on {d['dep_pl']}: {d['dep_sh']}% of xG"],
         f"#FPL #xG #GW{g} #FPLVORTEX"),
        ([f"\U0001F440 Who's beating their xG after GW{g}?",
          f"\u2705 {d['over']}: {d['over_g']} goals from {d['over_x']} xG",
          f"\u274C {d['under']}: {d['under_g']} goals from {d['under_x']} xG",
          f"\U0001F4C8 Top xG: {d['lead']} ({d['lead_xg']})",
          f"Full chart below \u2B07\uFE0F {d['channel']}"],
         f"#FPL #FantasyPremierLeague #GW{g} #FPLVORTEX"),
        ([f"\U0001F9E0 One-man attacks after GW{g}",
          f"{d['dep']} rely on {d['dep_pl']} for {d['dep_sh']}% of their xG \U0001F3AF",
          f"\U0001F525 Most xG: {d['lead']} ({d['lead_xg']})",
          f"\U0001F4CA Who creates each team's xG \u2193 | {d['channel']}"],
         f"#FPL #xG #GW{g} #FPLVORTEX"),
        ([f"\U0001F4C9 Regression watch | GW{g}",
          f"\U0001F9CA {d['under']} {d['under_g']} goals from {d['under_x']} xG",
          f"\U0001F525 {d['over']} {d['over_g']} goals from {d['over_x']} xG",
          "Small sample, so treat with care \U0001F9EA",
          f"{d['channel']} xG mix \u2193"],
         f"#FPL #xG #GW{g} #FPLVORTEX"),
        ([f"\U0001F6A8 {d['channel']} xG report | GW{g}",
          f"\U0001F947 Highest xG: {d['lead']} ({d['lead_xg']})",
          f"\U0001F3AF Biggest overperformer: {d['over']} ({d['over_d']})",
          f"\U0001F9CA Biggest underperformer: {d['under']} ({d['under_d']})"],
         f"#FPL #FantasyFootball #GW{g} #FPLVORTEX"),
    ]


def next_template_index(state: Path, count: int) -> int:
    """Cycle through every post angle before repeating; state persists in Drive."""
    try:
        previous = json.loads(state.read_text(encoding="utf-8"))
        queue = [int(item) for item in previous.get("remaining", []) if 0 <= int(item) < count]
        last = int(previous.get("last", -1))
    except (OSError, ValueError, TypeError):
        queue, last = [], -1
    if not queue:
        queue = list(range(count))
        random.shuffle(queue)
        if len(queue) > 1 and queue[0] == last:
            queue[0], queue[1] = queue[1], queue[0]
    choice = queue.pop(0)
    state.write_text(json.dumps({"last": choice, "remaining": queue}), encoding="utf-8")
    return choice


def make_post(shares, gw, channel, out_dir):
    """Pick a different template each run, trim to the X limit, return text."""
    d = _post_data(shares, gw, channel)
    temps = _templates(d)
    state = Path(out_dir) / "x_post_state.json"
    idx = next_template_index(state, len(temps))
    lines, tags = temps[idx]
    lines = list(lines)
    while x_len("\n".join(lines) + "\n\n" + tags) > X_LIMIT - 10 and len(lines) > 2:
        lines.pop()  # drop the least important line
    text = "\n".join(lines) + "\n\n" + tags
    return text


def save_post(text, out_dir, gw):
    f = Path(out_dir) / f"x_post_GW{gw}.txt"
    for old in Path(out_dir).glob("x_post_GW*.txt"):
        if old != f:
            old.unlink()
    f.write_text(text, encoding="utf-8")
    return f


# ------------------------------------------------------------- environment --
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def resolve_out_dir(arg):
    if arg:
        d = Path(arg)
    elif in_colab():
        from google.colab import drive
        if not Path("/content/drive/MyDrive").exists():
            drive.mount("/content/drive")
        d = Path(DRIVE_FOLDER)
    else:
        d = Path(".")
    d.mkdir(parents=True, exist_ok=True)
    return d


# -------------------------------------------------------------------- data --
def get_json(url, extra_headers=None):
    h = dict(HEADERS)
    if extra_headers:
        h.update(extra_headers)
    r = requests.get(url, headers=h, timeout=30)
    r.raise_for_status()
    return r.json()


def fpl_bootstrap():
    return get_json(FPL_URL)


def fpl_fixtures() -> list[dict]:
    fixtures = get_json("https://fantasy.premierleague.com/api/fixtures/")
    if not isinstance(fixtures, list):
        raise ValueError("FPL fixtures did not return a list")
    if len({fixture["id"] for fixture in fixtures}) != len(fixtures):
        raise ValueError("FPL fixtures contain duplicate match IDs")
    return fixtures


def fixture_has_ended(fixture: dict) -> bool:
    if not (fixture.get("finished") or fixture.get("finished_provisional")):
        return False
    if fixture.get("team_h_score") is None or fixture.get("team_a_score") is None:
        return False
    kickoff = fixture.get("kickoff_time")
    if not kickoff:
        return False
    kickoff_time = dt.datetime.fromisoformat(kickoff.replace("Z", "+00:00"))
    return kickoff_time <= dt.datetime.now(dt.timezone.utc)


def reporting_period(boot: dict, fixtures: list[dict]) -> dict:
    event_lookup = {int(event["id"]): event for event in boot["events"]}
    included = []
    latest = 0
    for event_id in sorted(event_lookup):
        games = [fixture for fixture in fixtures if fixture.get("event") == event_id]
        event = event_lookup[event_id]
        if not games:
            if event.get("finished") and event.get("data_checked"):
                latest = event_id
                continue
            break
        if not all(fixture_has_ended(fixture) for fixture in games):
            break
        included.extend(games)
        latest = event_id
    provisional = any(
        not event_lookup[int(fixture["event"])].get("data_checked")
        or not fixture.get("finished") for fixture in included
    )
    year = season_start_year(boot)
    return {
        "gameweek": latest,
        "range": f"GW1–GW{latest}" if latest else "No completed gameweek",
        "season": f"{year}–{str(year + 1)[2:]}",
        "fixtures": included,
        "fixture_count": len(included),
        "provisional": provisional,
        "status": "Provisional FPL data" if provisional else "FPL data checked",
        "cutoff_utc": max((fixture["kickoff_time"] for fixture in included), default=None),
        "retrieved_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "unassigned_fixtures": sum(fixture.get("event") is None for fixture in fixtures),
    }


def latest_finished_gw(boot: dict, fixtures: list[dict] | None = None) -> int:
    return reporting_period(boot, fixtures if fixtures is not None else fpl_fixtures())["gameweek"]


def period_metadata(period: dict) -> dict:
    return {key: value for key, value in period.items() if key != "fixtures"}


def _validate_event_statistics(payload: dict, games: list[dict], fields: tuple[str, ...]) -> dict[int, dict]:
    records = payload.get("elements")
    if not isinstance(records, list) or not records:
        raise ValueError("FPL gameweek statistics are missing")
    records_by_id = {int(record["id"]): record for record in records}
    if len(records_by_id) != len(records):
        raise ValueError("Duplicate FPL player statistics")
    appearances = set()
    fixture_ids = {int(fixture["id"]) for fixture in games}
    for record in records:
        stats = record.get("stats", {})
        for field in fields:
            value = stats.get(field)
            if value is None or not np.isfinite(float(value)) or float(value) < 0:
                raise ValueError(f"Missing/invalid {field} for player {record['id']}")
        for explanation in record.get("explain", []):
            if int(explanation["fixture"]) not in fixture_ids:
                raise ValueError("FPL live data includes a fixture outside its gameweek")
            if any(item.get("identifier") == "minutes" and float(item.get("value", 0)) > 0
                   for item in explanation.get("stats", [])):
                appearances.add(int(explanation["fixture"]))
    if appearances != fixture_ids:
        raise ValueError("Player statistics have not arrived for every completed fixture")
    for field in ("goals_scored", "defensive_contribution"):
        if field not in fields:
            continue
        expected = {}
        missing_fixture_stats = []
        for fixture in games:
            fixture_stats = fixture.get("stats", [])
            matching = [item for item in fixture_stats if item.get("identifier") == field]
            if not matching:
                if field == "defensive_contribution":
                    missing_fixture_stats.append(int(fixture["id"]))
                    continue
                if field == "goals_scored" and sum((fixture["team_h_score"], fixture["team_a_score"])) == 0:
                    continue
                raise ValueError(f"Fixture {fixture['id']} has no {field} validation data")
            for item in matching:
                for entry in item.get("h", []) + item.get("a", []):
                    player_id = int(entry["element"])
                    expected[player_id] = expected.get(player_id, 0) + float(entry["value"])
        if field == "defensive_contribution":
            # Fixture DEFCON is a cross-check; event/live is the report's player-level source.
            if not missing_fixture_stats:
                discrepancies = [
                    player_id for player_id in set(records_by_id) | set(expected)
                    if (float(records_by_id[player_id]["stats"][field]) if player_id in records_by_id else 0.0)
                    != expected.get(player_id, 0.0)
                ]
            else:
                discrepancies = []  # Incomplete fixture stats cannot establish a player-level mismatch.
            if discrepancies or missing_fixture_stats:
                warnings.warn(
                    f"FPL fixture/live DEFCON discrepancy: {len(discrepancies)} player(s) differ"
                    f"; {len(missing_fixture_stats)} fixture(s) lack a DEFCON breakdown. "
                    "Using FPL event/live defensive_contribution values for the report; "
                    "recheck when official data settles.",
                    RuntimeWarning,
                    stacklevel=2,
                )
            continue
        for player_id in set(records_by_id) | set(expected):
            record = records_by_id.get(player_id)
            actual = float(record["stats"][field]) if record else 0.0
            if actual != expected.get(player_id, 0.0):
                raise ValueError(f"Fixture/live {field} mismatch for player {player_id}")
    if "goals_scored" in fields:
        goals = sum(float(record["stats"]["goals_scored"]) for record in records)
        own_goals = sum(float(record["stats"].get("own_goals", 0)) for record in records)
        official_goals = sum(fixture["team_h_score"] + fixture["team_a_score"] for fixture in games)
        if goals + own_goals != official_goals:
            raise ValueError("FPL live goal totals have not caught up with final scores")
    if "expected_goals" in fields and sum(float(record["stats"]["expected_goals"]) for record in records) <= 0:
        raise ValueError("Completed fixtures have no published xG data")
    return records_by_id


def cumulative_player_bootstrap(boot: dict, period: dict, fields: tuple[str, ...], cache_dir: Path) -> dict:
    if not period["gameweek"]:
        raise ValueError("No complete gameweek available")
    event_ids = sorted({int(fixture["event"]) for fixture in period["fixtures"]})
    cache_dir.mkdir(parents=True, exist_ok=True)
    def fetch_event(event_id: int) -> tuple[int, dict]:
        url = f"https://fantasy.premierleague.com/api/event/{event_id}/live/"
        payload = get_json(url)
        games = [fixture for fixture in period["fixtures"] if fixture["event"] == event_id]
        _validate_event_statistics(payload, games, fields)
        return event_id, payload
    totals = {int(player["id"]): {field: 0.0 for field in fields} for player in boot["elements"]}
    with ThreadPoolExecutor(max_workers=4) as executor:
        for event_id, payload in executor.map(fetch_event, event_ids):
            for record in payload["elements"]:
                player_totals = totals.get(int(record["id"]))
                if player_totals is None:
                    if any(float(record["stats"][field]) > 0 for field in fields):
                        raise ValueError(f"Historical player {record['id']} missing from bootstrap")
                    continue
                for field in fields:
                    player_totals[field] += float(record["stats"][field])
            (cache_dir / f"{season_start_year(boot)}_GW{event_id}.json").write_text(
                json.dumps(payload), encoding="utf-8")
    limited_boot = dict(boot)
    limited_boot["elements"] = [dict(player, **totals[int(player["id"])]) for player in boot["elements"]]
    limited_boot["report_period"] = period_metadata(period)
    return limited_boot


def _save_report_metadata(path: Path, period: dict, figures: dict) -> None:
    metadata = dict(period_metadata(period), **figures)
    path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def _fit_text_width(text_artist, left: float, right: float, minimum_size: float = 15.0,
                    ellipsis: bool = False) -> None:
    ax = text_artist.axes
    renderer = ax.figure.canvas.get_renderer()
    available = abs(ax.transData.transform((right, 0))[0] - ax.transData.transform((left, 0))[0])
    while text_artist.get_window_extent(renderer).width > available and text_artist.get_fontsize() > minimum_size:
        text_artist.set_fontsize(max(minimum_size, text_artist.get_fontsize() - 0.5))
    while ellipsis and text_artist.get_window_extent(renderer).width > available and len(text_artist.get_text()) > 3:
        text_artist.set_text(text_artist.get_text().rstrip("…")[:-1] + "…")
    if text_artist.get_window_extent(renderer).width > available + 1:
        raise ValueError(f"Text does not fit its column: {text_artist.get_text()}")


def _save_native_report(fig, path: str, period_label: str) -> None:
    width, height = fig.get_size_inches()
    dpi = 3840 / min(width, height)
    fig.set_dpi(dpi)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    bounds = fig.bbox
    for ax in fig.axes:
        for artist in ax.texts:
            extent = artist.get_window_extent(renderer)
            if extent.x0 < -1 or extent.x1 > bounds.width + 1 or extent.y0 < -1 or extent.y1 > bounds.height + 1:
                plt.close(fig)
                raise ValueError(f"Report text leaves the canvas: {artist.get_text()}")
    fig.savefig(path, dpi=dpi, facecolor=BG, metadata={"Description": period_label})
    plt.close(fig)


def _fit_x_caption(candidates: list[str]) -> str:
    for candidate in candidates:
        if x_len(candidate) <= X_LIMIT:
            return candidate
    raise ValueError("The data-driven caption exceeds X's 280-character limit")


def season_start_year(boot):
    return int(boot["events"][0]["deadline_time"][:4])


def teams_table(boot):
    return pd.DataFrame(boot["teams"])[["id", "code", "name", "short_name"]]


def mix_fpl_position(boot):
    """Team xG by position, plus xG total, goals and top contributor."""
    el = pd.DataFrame(boot["elements"])
    el["xg"] = pd.to_numeric(el["expected_goals"], errors="coerce").fillna(0.0)
    el["goals"] = pd.to_numeric(el["goals_scored"], errors="coerce").fillna(0.0)
    piv = el.pivot_table(index="team", columns="element_type", values="xg",
                         aggfunc="sum", fill_value=0.0)
    for t in (1, 2, 3, 4):
        if t not in piv.columns:
            piv[t] = 0.0
    piv = piv.rename(columns={1: "goalkeepers", 2: "defenders",
                              3: "midfielders", 4: "forwards"})
    agg = el.groupby("team").agg(xg_total=("xg", "sum"), goals=("goals", "sum"))
    top = (el.sort_values("xg", ascending=False).groupby("team").head(1)
           .set_index("team"))
    piv = piv.join(agg)
    piv["top_player"] = top["web_name"]
    piv["top_share"] = (top["xg"] / piv["xg_total"].replace(0, np.nan) * 100).fillna(0)
    teams = teams_table(boot).set_index("id")
    piv.insert(0, "team", teams.loc[piv.index, "name"].values)
    return piv.reset_index(drop=True), CATS_POSITION


def _match_fpl_name(name, fpl_names):
    if name in ALIASES and ALIASES[name] in fpl_names:
        return ALIASES[name]
    if name in fpl_names:
        return name
    m = difflib.get_close_matches(name, fpl_names, n=1, cutoff=0.6)
    return m[0] if m else None


def mix_understat(boot):
    season = season_start_year(boot)
    xhr = {"X-Requested-With": "XMLHttpRequest"}
    league = get_json(f"https://understat.com/getLeagueData/EPL/{season}", xhr)
    names = [t["title"] for t in league["teams"].values()]
    fpl_names = teams_table(boot)["name"].tolist()
    rows, unmatched = [], []
    for n in names:
        fpl_name = _match_fpl_name(n, fpl_names)
        if fpl_name is None:
            unmatched.append(n)
            continue
        data = get_json(
            f"https://understat.com/getTeamData/{n.replace(' ', '_')}/{season}", xhr)
        sit = data["statistics"]["situation"]
        row = {"team": fpl_name}
        for k, col in UNDERSTAT_MAP.items():
            row[col] = float(sit.get(k, {}).get("xG", 0) or 0)
        rows.append(row)
    if unmatched:
        raise RuntimeError(f"Could not match Understat teams to FPL names: "
                           f"{unmatched}. Add them to ALIASES.")
    return pd.DataFrame(rows), CATS_SITUATION


def prepare(df, cats, sort="xg"):
    df = df.copy()
    cols_all = [c[0] for c in cats]
    for c in cols_all:
        if c not in df.columns:
            df[c] = 0.0
    extra = [c for c in df.columns if c not in cols_all + ["team"]]
    df[cols_all] = df[cols_all].fillna(0).astype(float)
    cats = [c for c in cats if df[c[0]].sum() > 0]
    if not cats:
        raise ValueError("All xG categories are zero; nothing to chart yet.")
    cols = [c[0] for c in cats]
    total = df[cols].sum(axis=1)
    shares = df[cols].div(total.replace(0, np.nan), axis=0).fillna(0) * 100
    shares.insert(0, "team", df["team"].values)
    for c in extra:
        shares[c] = df[c].values
    has_extra = {"xg_total", "goals"}.issubset(shares.columns)
    if has_extra:
        shares["diff"] = shares["goals"] - shares["xg_total"]
    key = cols[0]
    if has_extra and sort == "xg":
        key = "xg_total"
    elif has_extra and sort == "diff":
        key = "diff"
    shares = shares.sort_values(key, ascending=False).reset_index(drop=True)
    return shares, cats


# ------------------------------------------------------------------ badges --
def fetch_logo(code, cache_dir):
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    f = cache / f"t{code}.png"
    if not f.exists():
        for url in BADGE_URLS:
            try:
                r = requests.get(url.format(code=code), headers=HEADERS, timeout=20)
                if r.ok and r.headers.get("content-type", "").startswith("image"):
                    f.write_bytes(r.content)
                    break
            except requests.RequestException:
                continue
    if not f.exists():
        return None
    try:
        return np.asarray(Image.open(f).convert("RGBA"))
    except Exception:
        return None


def load_logos(boot, cache_dir):
    return {r["name"]: fetch_logo(r["code"], cache_dir)
            for _, r in teams_table(boot).iterrows()}


# ------------------------------------------------------------ headshots / UI --
PLAYER_PHOTO_URLS = (
    "https://resources.premierleague.com/premierleague25/photos/players/110x140/{code}.png",
    "https://resources.premierleague.com/premierleague/photos/players/110x140/p{code}.png",
)


def top_three_by_team(boot, metric="expected_goals"):
    """Preserves player identity from FPL bootstrap rather than matching on names."""
    elements = pd.DataFrame(boot["elements"]).copy()
    if metric not in elements.columns:
        raise ValueError(f"FPL bootstrap has no {metric!r} column")
    elements["metric"] = pd.to_numeric(elements[metric], errors="coerce").fillna(0.0)
    elements = elements[elements["element_type"].isin((2, 3, 4))].copy()
    elements = elements.sort_values(["metric", "id"], ascending=[False, True])
    by_id = teams_table(boot).set_index("id")["name"].to_dict()
    role_names = {1: "goalkeepers", 2: "defenders", 3: "midfielders", 4: "forwards"}
    result = {}
    for team_id, group in elements.groupby("team", sort=False):
        result[by_id[int(team_id)]] = [
            {"name": str(rec.web_name), "value": float(rec.metric),
             "code": str(getattr(rec, "code", "")),
             "photo": str(getattr(rec, "photo", "")),
             "role": role_names[int(rec.element_type)]}
            for rec in group.head(3).itertuples(index=False)
        ]
    return result


def fetch_player_photo(player, cache_dir):
    image_code = re.sub(r"\.(?:png|jpg|jpeg)$", "", player.get("photo", ""), flags=re.I)
    if not image_code.isdigit():
        image_code = str(player.get("code", ""))
    if not image_code.isdigit():
        return None
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cached = cache_dir / f"player_{image_code}.png"
    if cached.exists():
        try:
            return np.asarray(Image.open(cached).convert("RGBA"))
        except (OSError, ValueError):
            cached.unlink(missing_ok=True)
    for pattern in PLAYER_PHOTO_URLS:
        try:
            response = requests.get(pattern.format(code=image_code), headers=HEADERS, timeout=9)
            response.raise_for_status()
            if not response.headers.get("content-type", "").lower().startswith("image/"):
                continue
            with Image.open(io.BytesIO(response.content)) as image:
                rgba = image.convert("RGBA")
                rgba.save(cached)
                return np.asarray(rgba)
        except (OSError, ValueError, requests.RequestException):
            continue
    return None


def load_player_photos(top_players, cache_dir, workers=10):
    players = {str(p.get("photo") or p.get("code")): p
               for team in top_players.values() for p in team}
    photographs = {}
    with ThreadPoolExecutor(max_workers=workers) as executor:
        futures = {executor.submit(fetch_player_photo, p, cache_dir): key
                   for key, p in players.items()}
        for future in as_completed(futures):
            key = futures[future]
            try:
                photographs[key] = future.result()
            except Exception as exc:
                print(f"Portrait unavailable for {key}: {exc}")
                photographs[key] = None
    loaded = sum(picture is not None for picture in photographs.values())
    print(f"Player portraits cached/loaded: {loaded}/{len(players)}; missing ones use initials.")
    return photographs


def add_picture(ax, array, x, y, size=.44):
    if array is None:
        ax.scatter([x], [y], s=540, facecolors="#D5DCE7", edgecolors="#8993A8", zorder=3)
        return
    zoom = size * 110.0 / max(array.shape[:2])
    ax.add_artist(AnnotationBbox(OffsetImage(array, zoom=zoom), (x, y),
                                 frameon=False, box_alignment=(.5, .5), zorder=5))


def crop_headshot(array):
    if array is None:
        return None
    h, w = array.shape[:2]
    x0 = int(w * 0.16)
    x1 = int(w * 0.84)
    y0 = int(h * 0.02)
    y1 = int(h * 0.58)
    if x1 <= x0 or y1 <= y0:
        return array
    return array[y0:y1, x0:x1]


def social_footer(ax, y, label, x_handle=X_HANDLE, yt_handle=YOUTUBE_HANDLE):
    T(ax, 50, y + .37, label, fontsize=10.3, font="semi", color="#536074",
      ha="center", va="center")
    # Vector X prevents a missing-glyph square when optional fonts are unavailable.
    ax.plot([34.28, 35.05], [y + .14, y - .14], color=INK, lw=3, solid_capstyle="round")
    ax.plot([34.32, 35.00], [y - .14, y + .14], color=INK, lw=1.7, solid_capstyle="round")
    T(ax, 37, y, x_handle, fontsize=11, color=INK, font="bold", va="center")
    youtube = FancyBboxPatch((57, y - .14), 2.0, .32,
                              boxstyle="round,pad=0.05,rounding_size=0.08",
                              color="#ED1D24", lw=0)
    ax.add_patch(youtube)
    ax.add_patch(Polygon([[57.72, y - .085], [57.72, y + .115], [58.4, y + .015]],
                         closed=True, facecolor="white", edgecolor="white"))
    T(ax, 60, y, yt_handle, fontsize=11, color=INK, font="bold", va="center")


def headline(ax, fig, H, title, subtitle, channel, brand_logo):
    T(ax, 53, H - .55, title.upper(), ha="center", va="center",
      fontsize=36, font="title", color=INK)
    T(ax, 53, H - 1.16, subtitle, ha="center", va="center",
      fontsize=11.8, color="#374151")
    if brand_logo and Path(brand_logo).exists():
        with Image.open(brand_logo) as image:
            ratio = image.width / max(image.height, 1)
        logo_h = 1.25
        logo_w = min(logo_h * ratio, 2.6)
        logo_ax = fig.add_axes([.019, 1 - (0.22 + logo_h) / H,
                               logo_w / 20, logo_h / H])
        logo_ax.imshow(plt.imread(str(brand_logo)))
        logo_ax.axis("off")
        name_x = 2.5 + logo_w * 5
        T(ax, name_x, H - .82, channel, fontsize=19, font="bold", color=BRAND_BG,
          va="center", ha="left")
    else:
        T(ax, 3.4, H - .78, "◈", fontsize=23, color=GOOD, ha="center", va="center")
        T(ax, 6.6, H - .78, channel, fontsize=19, font="bold", color=BRAND_BG,
          va="center", ha="left")


def _block(ax, cx, y, heading, colour, lines):
    T(ax, cx, y, heading, ha="center", va="center", fontsize=12.2,
      font="bold", color=colour)
    for j, content in enumerate(lines):
        T(ax, cx, y - .32 - j * .27, content, ha="center", va="center",
          fontsize=9.6, color=INK)


def draw(shares, cats, title, subtitle, footer, out, logos=None, short=None,
         channel=CHANNEL, brand_logo=None, note=None, top_players=None, photos=None,
         mode="xg"):
    logos, short = logos or {}, short or {}
    top_players, photos = top_players or {}, photos or {}
    is_xg = mode == "xg" and {"xg_total", "goals", "diff"}.issubset(shares.columns)
    is_defcon = mode == "defcon"
    n = len(shares)
    W, row_h, top_h, bot_h = 20, .80, 2.35, 4.25
    H = top_h + n * row_h + bot_h
    fig = plt.figure(figsize=(W, H), facecolor=BG)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set(xlim=(0, 100), ylim=(0, H))
    ax.axis("off")
    headline(ax, fig, H, title, subtitle, channel, brand_logo)

    bar_x0, bar_x1 = 23.0, 47.5
    hdr_y = H - top_h + .04
    for x, caption, align in [(2.7, "#", "center"), (11, "TEAM", "left"),
                               (35.3, "MIX (%)", "center")]:
        T(ax, x, hdr_y, caption, ha=align, va="center", fontsize=12,
          font="bold", color=INK)
    if is_xg:
        numeric_headers = [(53.0, "xG"), (59.0, "GOALS"), (65.0, "G−xG")]
        player_heading = "TOP 3 xG CONTRIBUTORS"
    elif is_defcon:
        numeric_headers = [(55.4, "ACTIONS"), (63.4, "/90")]
        player_heading = "TOP 3 DEFCON CONTRIBUTORS"
    else:
        numeric_headers = []
        player_heading = "TOP 3 xG CONTRIBUTORS (FPL)"
    for x, label in numeric_headers:
        T(ax, x, hdr_y, label, ha="center", va="center", fontsize=12,
          font="bold", color=INK)
    T(ax, 83.5, hdr_y, player_heading, ha="center", va="center",
      fontsize=12, font="bold", color=INK)
    ax.plot([1, 99], [hdr_y - .28] * 2, color=GRID, lw=1.7)

    for idx, (_, rec) in enumerate(shares.iterrows()):
        y = hdr_y - .33 - row_h * (idx + .5)
        if idx % 2 == 0:
            ax.add_patch(Rectangle((1, y - row_h / 2), 98, row_h,
                                   color="#EDE8DE", alpha=.53, lw=0))
        team = str(rec["team"])
        T(ax, 2.7, y, str(idx + 1), ha="center", va="center", fontsize=11.4, color=INK)
        badge = logos.get(team)
        if badge is not None:
            add_picture(ax, badge, 7.6, y, .34)
        else:
            ax.scatter([7.6], [y], s=470, color="#CFD3DA")
            T(ax, 7.6, y, short.get(team, team[:3])[:3].upper(), ha="center",
              va="center", fontsize=7, font="bold", color=INK)
        T(ax, 11, y, team[:17], ha="left", va="center", fontsize=11.3, color=INK)
        left = bar_x0
        for column, _, color in cats:
            percentage = float(rec[column])
            width = max(0.0, percentage) / 100.0 * (bar_x1 - bar_x0)
            if width <= 0:
                continue
            ax.add_patch(Rectangle((left, y - .16), width, .32, lw=0, color=color))
            if percentage >= 12:
                T(ax, left + width / 2, y, f"{percentage:.0f}%", ha="center", va="center",
                  fontsize=9.4, font="bold", color="white")
            left += width
        if is_xg:
            T(ax, 53, y, f"{rec['xg_total']:.1f}", ha="center", va="center", fontsize=11.2)
            T(ax, 59, y, str(int(round(rec["goals"]))), ha="center", va="center", fontsize=11.2)
            T(ax, 65, y, f"{rec['diff']:+.1f}", ha="center", va="center",
              fontsize=11.2, font="bold", color=GOOD if rec["diff"] >= 0 else BAD)
        elif is_defcon:
            T(ax, 55.4, y, str(int(round(rec["actions"]))),
              ha="center", va="center", fontsize=11.2)
            T(ax, 63.4, y, f"{rec['actions90']:.1f}",
              ha="center", va="center", fontsize=11.2)
        for rank, player in enumerate(top_players.get(team, [])[:3]):
            start = (69.2, 79.3, 89.4)[rank]
            image_key = str(player.get("photo") or player.get("code"))
            picture = photos.get(image_key)
            add_picture(ax, picture, start + 1, y, .50)
            text_name = str(player.get("name", ""))
            if len(text_name) > 12:
                text_name = text_name[:11] + "…"
            T(ax, start + 2.8, y + .16, text_name, ha="left", va="center",
              fontsize=8.6, font="semi", color=INK)
            unit = "xG" if not is_defcon else "actions"
            value = float(player.get("value", 0))
            value_label = f"{value:.2f} {unit}" if unit == "xG" else f"{int(value)} {unit}"
            T(ax, start + 2.8, y - .15, value_label, ha="left", va="center",
              fontsize=8.2, color="#526176")
        ax.plot([1.0, 99.0], [y - row_h / 2] * 2, color=GRID, lw=.65)

    legend_y = hdr_y - .33 - n * row_h - .50
    step = 96 / len(cats)
    for index, (_, label, color) in enumerate(cats):
        x = 3 + step * index
        ax.add_patch(Rectangle((x, legend_y - .09), 1.2, .18, color=color, lw=0))
        T(ax, x + 1.7, legend_y, label, fontsize=10.8, ha="left", va="center", color=INK)
    block_y = legend_y - .55
    if is_xg:
        dep = shares.nlargest(3, "top_share")
        top = shares.nlargest(3, "diff")
        under = shares.nsmallest(3, "diff")
        _block(ax, 17, block_y, "RELIANT ON ONE PLAYER", GOOD,
               [f"{r.team}: {str(r.top_player)} ({r.top_share:.0f}%)" for r in dep.itertuples()])
        _block(ax, 50, block_y, "BEATING xG", GOOD,
               [f"{r.team}: {r.diff:+.1f} G−xG" for r in top.itertuples()])
        _block(ax, 83, block_y, "BELOW xG", BAD,
               [f"{r.team}: {r.diff:+.1f} G−xG" for r in under.itertuples()])
    elif is_defcon:
        leaders = shares.nlargest(3, "actions")
        rates = shares.nlargest(3, "actions90")
        players = sorted([dict(player, team=team) for team, plist in top_players.items()
                          for player in plist], key=lambda p: p["value"], reverse=True)[:3]
        _block(ax, 17, block_y, "MOST DEFCON ACTIONS", GOOD,
               [f"{r.team}: {int(r.actions)}" for r in leaders.itertuples()])
        _block(ax, 50, block_y, "ACTIONS PER 90", "#3B82F6",
               [f"{r.team}: {r.actions90:.1f}" for r in rates.itertuples()])
        _block(ax, 83, block_y, "TOP PLAYERS", "#8E68D6",
               [f"{p['name']} ({p['team']}): {int(p['value'])}" for p in players])
    else:
        non_other = cats[:3]
        for index, (column, label, color) in enumerate(non_other):
            entries = shares.nlargest(3, column)
            _block(ax, 100 / len(non_other) * (index + .5), block_y, label.upper(), color,
                   [f"{r.team}: {getattr(r, column):.1f}%" for r in entries.itertuples()])
    if note:
        T(ax, 50, 1.42, note, ha="center", va="center", fontsize=9,
          color="#5E6670")
    social_footer(ax, .30, footer)
    fig.savefig(out, dpi=150, facecolor=BG, bbox_inches="tight", pad_inches=.08)
    plt.close(fig)
    print(f"Saved {out}")

# -------------------------------------------------------------------- main --
def run(args):
    root_dir = resolve_out_dir(args.out_dir)
    out_dir = root_dir / "xG"
    out_dir.mkdir(parents=True, exist_ok=True)
    asset_dir = root_dir / "_assets"
    badge_dir = asset_dir / "badges"

    boot = fpl_bootstrap()
    gw = latest_finished_gw(boot)
    if gw == 0:
        print("No finished gameweek yet - nothing to chart.")
        return None
    out = out_dir / f"{args.prefix}_GW{gw}.png"
    if args.only_if_new_gw and out.exists():
        print(f"GW{gw} already rendered - nothing to do.")
        return out

    if args.source == "fpl":
        df, cats = mix_fpl_position(boot)
        title, what, src = ("Who Creates Each Team's xG",
                            "Share of team xG by player position", "FPL API")
    elif args.source == "understat":
        df, cats = mix_understat(boot)
        title, what, src = ("How Teams Create xG",
                            "Share of total xG by situation", "Understat")
    else:
        if not args.csv:
            raise SystemExit("--source csv needs --csv FILE")
        df, cats = pd.read_csv(args.csv), CATS_SITUATION
        title, what, src = ("How Teams Create xG",
                            "Share of total xG by situation", args.csv_source)

    shares, cats = prepare(df, cats, args.sort)
    if len(shares) != 20:
        print(f"WARNING: {len(shares)} teams in data (expected 20).")

    t = teams_table(boot)
    short = dict(zip(t["name"], t["short_name"]))
    logos = {} if args.no_logos else load_logos(boot, badge_dir)
    players = top_three_by_team(boot, "expected_goals")
    photos = {} if args.no_photos else load_player_photos(players, asset_dir / "player_photos")

    y0 = season_start_year(boot)
    subtitle = f"{what} | Premier League {y0}-{str(y0 + 1)[2:]} | after GW{gw}"
    footer = f"{args.channel} | Data via {src} | updated {dt.date.today():%d.%m.%Y}"
    note = None
    if "xg_total" in shares.columns:
        note = ("G\u2212xG = goals minus xG. xG and goals are credited to a player's "
                "current club. Small samples regress toward zero.")
    load_fonts(asset_dir / "fonts")
    brand_logo = resolve_logo(args.logo, asset_dir)
    draw(shares, cats, title, subtitle, footer, str(out), logos, short,
         args.channel, brand_logo, note, players, photos, mode="xg")
    clean_old(out_dir, args.prefix, out)
    if not args.no_post:
        post = make_post(shares, gw, args.channel, out_dir)
        f = save_post(post, out_dir, gw)
        print(f"\n--- X post ({x_len(post)}/{X_LIMIT}) saved to {f.name} ---\n{post}\n")
    return out


def parse(argv=None):
    p = argparse.ArgumentParser(description="FPL VORTEX xG mix chart")
    p.add_argument("--source", choices=["fpl", "understat", "csv"], default="fpl")
    p.add_argument("--csv")
    p.add_argument("--csv-source", default="Opta")
    p.add_argument("--sort", choices=["xg", "share", "diff"], default="xg",
                   help="xg = total xG, share = first category share, diff = G-xG")
    p.add_argument("--channel", default=CHANNEL)
    p.add_argument("--logo", default=DEFAULT_LOGO,
                   help="Drive link / file id / local path, or 'none'")
    p.add_argument("--prefix", default="pl_xg_mix", help="chart file name prefix")
    p.add_argument("--out-dir", default=None, help="folder (default: Drive in Colab)")
    p.add_argument("--no-logos", action="store_true")
    p.add_argument("--no-photos", action="store_true", help="skip player photo downloads")
    p.add_argument("--no-post", action="store_true", help="skip the X post text")
    p.add_argument("--only-if-new-gw", action="store_true")
    args, _ = p.parse_known_args(argv)  # ignores Colab's hidden -f argument
    return args


def main(argv=None):
    out = run(parse(argv))
    if out is not None and in_colab():
        from IPython.display import Image as IPImage, display
        display(IPImage(str(out)))
    return out





# ========================= V3 OUTPUT OVERRIDES: 1 / 1A / 1B =========================
from IPython.display import Image as IPImage, display


def _truncate_name(name, limit=12):
    name = str(name)
    return name if len(name) <= limit else name[:limit - 1] + "…"


def clear_report_folder(out_dir: Path | str) -> None:
    report_dir = Path(out_dir)
    report_dir.mkdir(parents=True, exist_ok=True)
    # Keep the last full-page report until its replacement is generated.
    for old_name in ("1A.png", "1B.png", "1A.txt", "1B.txt"):
        old_path = report_dir / old_name
        if old_path.is_file() and not old_path.is_symlink():
            old_path.unlink()


def _output_paths(out_dir):
    return {"1": Path(out_dir) / "1.png"}


def _text_paths(out_dir):
    return {"1": Path(out_dir) / "1.txt"}


def _xg_post_for_section(segment, gw, channel, label, season=None, provisional=False):
    lead = segment.nlargest(1, "xg_total").iloc[0]
    over = segment.nlargest(1, "diff").iloc[0]
    under = segment.nsmallest(1, "diff").iloc[0]
    status = " | provisional" if provisional else ""
    heading = f"{channel} | xG {season or ''} | GW1–GW{gw}{status}".replace("  ", " ")
    insights = (f"{lead['team']}: {lead['xg_total']:.1f} xG.\n"
                f"{over['team']}: {over['goals']:.0f} goals vs {over['xg_total']:.1f} xG ({over['diff']:+.1f}).\n"
                f"{under['team']}: {under['diff']:+.1f} goals minus xG.")
    tags = "#FPL #ExpectedGoals #PremierLeague"
    return _fit_x_caption([
        f"{heading}\n{insights}\nChance quality vs finishing for Fantasy Premier League.\n{tags}",
        f"{heading}\n{insights}\nFPL chance quality vs finishing.\n{tags}",
        f"{heading}\n{insights}\nFPL xG measures chance quality.\n#FPL #ExpectedGoals",
    ])


def _defcon_post_for_section(segment, gw, channel, label, season=None, top_players=None, provisional=False):
    lead = segment.nlargest(1, "actions").iloc[0]
    rate = segment.nlargest(1, "actions90").iloc[0]
    ranked_players = sorted(
        (dict(player, team=team) for team, players in (top_players or {}).items() for player in players),
        key=lambda player: (-float(player["value"]), str(player["name"])),
    )
    player_line = (f"{ranked_players[0]['name']}: {ranked_players[0]['value']:.0f} actions."
                   if ranked_players else f"{segment.nsmallest(1, 'actions').iloc[0]['team']}: fewest actions.")
    status = " | provisional" if provisional else ""
    heading = f"{channel} | DEFCON {season or ''} | GW1–GW{gw}{status}".replace("  ", " ")
    insights = (f"{lead['team']}: {lead['actions']:.0f} actions; "
                f"{rate['team']}: {rate['actions90']:.1f}/90 player-minutes.\n{player_line}")
    tags = "#FPL #DEFCON #PremierLeague"
    context = "Defensive workload for Fantasy Premier League; actions, not points."
    return _fit_x_caption([
        f"{heading}\n{insights}\n{context}\n{tags}",
        f"{heading}\n{insights}\nFPL defensive workload; actions, not points.\n{tags}",
        f"{heading}\n{lead['team']}: {lead['actions']:.0f} actions. "
        f"{rate['team']}: {rate['actions90']:.1f}/90 player-minutes.\n{player_line}\nActions, not FPL points.\n#FPL #DEFCON",
    ])


def _save_text(path, text):
    path = Path(path)
    path.write_text(text, encoding="utf-8")
    return path


def _draw_brand_header(ax, fig, H, title, subtitle, channel, brand_logo, cfg):
    # Keep branding on its own line above the report title, outside the logo footprint.
    width = float(cfg.get("W", 24.0))
    brand_x = 5.8
    if brand_logo and Path(brand_logo).exists():
        with Image.open(brand_logo) as image:
            ratio = image.width / max(image.height, 1)
        logo_h = cfg.get("logo_h", 1.25)
        logo_w = min(logo_h * ratio, cfg.get("logo_w_max", 2.5))
        logo_x = 0.017
        logo_ax = fig.add_axes([logo_x, 1 - (0.16 + logo_h) / H,
                                logo_w / width, logo_h / H])
        logo_ax.imshow(plt.imread(str(brand_logo)))
        logo_ax.axis("off")
        logo_ax.set_zorder(4)
        brand_x = logo_x * 100 + logo_w / width * 100 + 1.6
    else:
        ax.text(3.0, H - 0.72, "◈", fontsize=24, color=GOOD, ha="center", va="center",
                fontfamily="DejaVu Sans")

    T(ax, brand_x, H - 0.72, channel, fontsize=cfg.get("brand_fs", 23),
      font="bold", color=BRAND_BG, va="center", ha="left", zorder=7)
    title_x = cfg.get("title_x", 52)
    T(ax, title_x, H - 1.68, title.upper(), ha="center", va="center",
      fontsize=cfg.get("title_fs", 43), font="title", color=INK)
    T(ax, title_x, H - 2.40, subtitle, ha="center", va="center",
      fontsize=cfg.get("subtitle_fs", 14), color="#374151")


def _draw_social_footer(ax, y, label, cfg, x_handle=X_HANDLE, yt_handle=YOUTUBE_HANDLE):
    T(ax, 50, y + 0.42, label, fontsize=cfg.get("footer_label_fs", 10.8),
      font="semi", color="#536074", ha="center", va="center")
    x0 = 34.1
    ax.plot([x0, x0 + 0.85], [y + 0.17, y - 0.17], color=INK, lw=3.2, solid_capstyle="round")
    ax.plot([x0 + 0.05, x0 + 0.78], [y - 0.17, y + 0.17], color=INK, lw=1.8, solid_capstyle="round")
    T(ax, 37.0, y, x_handle, fontsize=cfg.get("social_fs", 11.5), color=INK, font="bold", va="center")
    youtube = FancyBboxPatch((57.0, y - 0.16), 2.12, 0.36,
                             boxstyle="round,pad=0.05,rounding_size=0.08",
                             color="#ED1D24", lw=0)
    ax.add_patch(youtube)
    ax.add_patch(Polygon([[57.76, y - 0.09], [57.76, y + 0.12], [58.47, y + 0.015]],
                         closed=True, facecolor="white", edgecolor="white"))
    T(ax, 60.4, y, yt_handle, fontsize=cfg.get("social_fs", 11.5), color=INK, font="bold", va="center")


def add_ranked_headshot(ax, picture, x, y, row_h):
    if picture is None:
        ax.scatter([x], [y], s=690, facecolors="#D5DCE7", edgecolors="#8993A8", zorder=4)
        return
    height_px, width_px = picture.shape[:2]
    row_fit = row_h * 72.0 * 0.90 / max(height_px, 1)
    col_fit = 4.2 * ax.figure.get_size_inches()[0] / 100.0 * 72.0 / max(width_px, 1)
    zoom = min(row_fit, col_fit)
    ax.add_artist(AnnotationBbox(OffsetImage(picture, zoom=zoom), (x, y),
                                 frameon=False, box_alignment=(0.5, 0.5), zorder=5))

import base64
import math
import shutil
import subprocess
import sys
from html import escape as html_escape
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from typing import Any
from PIL.PngImagePlugin import PngInfo
from playwright.sync_api import sync_playwright

HTML_DESIGN_WIDTH = 1440
PNG_EXPORT_WIDTH = 3840

REPORT_CSS = r'''
* { box-sizing: border-box; }
html, body { margin: 0; padding: 0; background: #F4EFE4; color: #111827; }
body { font-family: Poppins, Arial, sans-serif; font-size: 26px; }
.report { width: 100%; max-width: 1440px; margin: 0 auto; padding: 28px 32px 30px; }
.brand { display: flex; align-items: center; gap: 26px; margin-bottom: 20px; }
.brand-logo { display: block; width: min(320px, 36vw); height: auto; object-fit: contain; }
.brand-name { font-size: clamp(32px, 2.64vw, 38px); font-weight: 700; color: #0B1B3A; }
.brand-fallback { font-size: 68px; color: #1E8E4E; line-height: 1; }
h1 { margin: 0 0 12px; font-family: BebasNeue, Impact, sans-serif;
     font-size: clamp(48px, 4.167vw, 60px); line-height: 1.08; font-weight: 400;
     text-align: center; overflow-wrap: anywhere; }
.subtitle { margin: 0 auto 24px; font-size: clamp(28px, 2.222vw, 32px);
            line-height: 1.25; text-align: center; color: #374151; }
table { width: 100%; border-collapse: collapse; table-layout: fixed; }
th, td { padding: 6px 4px; vertical-align: middle; border-bottom: 1px solid #D9D2C3;
         word-wrap: break-word; overflow-wrap: anywhere; }
th { padding-top: 12px; padding-bottom: 12px; font-size: 26px; font-weight: 700;
     line-height: 1.2; text-align: center; border-bottom-width: 2px; }
td { font-size: 26px; line-height: 1.2; }
tr:nth-child(odd) td { background: #EDE8DE; }
.rank { text-align: center; font-size: 22px; }
.team { display: flex; align-items: center; gap: 10px; min-width: 0; }
.team-badge { width: 42px; height: 46px; flex: 0 0 42px; object-fit: contain; }
.team-fallback { width: 42px; flex: 0 0 42px; font-size: 20px; font-weight: 700; text-align: center; }
.team-name { min-width: 0; font-size: 26px; line-height: 1.2; }
.metric { text-align: center; font-variant-numeric: tabular-nums; }
.metric-positive { color: #1E8E4E; font-weight: 700; }
.metric-negative { color: #C0392B; font-weight: 700; }
.mix-track { display: flex; width: 100%; height: 24px; }
.mix-segment { display: block; flex: 0 0 auto; }
.mix-labels { display: grid; grid-template-columns: repeat(var(--category-count), minmax(0, 1fr));
              gap: 4px; margin-top: 4px; text-align: center; }
.mix-pct { font-size: 20px; font-weight: 700; line-height: 1.2; }
.contributors { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr)); gap: 8px; }
.player { display: flex; align-items: center; gap: 6px; min-width: 0; }
.player-photo { display: block; flex: 0 0 36px; width: 36px; height: 50px; object-fit: contain; }
.player-fallback { flex: 0 0 36px; width: 36px; font-size: 20px; text-align: center; color: #536074; }
.player-copy { min-width: 0; flex: 1; }
.player-name { font-size: 20px; font-weight: 700; line-height: 1.16; word-wrap: break-word;
               overflow-wrap: anywhere; }
.player-value { margin-top: 4px; font-size: 22px; font-weight: 700; line-height: 1.18;
                overflow-wrap: anywhere; font-variant-numeric: tabular-nums; }
.legend { display: flex; flex-wrap: wrap; justify-content: space-between; gap: 12px 24px;
          padding: 18px 4px 8px; font-size: 22px; }
.legend-item { display: inline-flex; align-items: center; gap: 10px; }
.swatch { width: 20px; height: 12px; flex: 0 0 20px; }
.summaries { display: grid; grid-template-columns: repeat(3, minmax(0, 1fr));
             gap: 24px; margin-top: 18px; text-align: center; }
.summary h2 { margin: 0 0 9px; font-size: 24px; line-height: 1.35; font-weight: 700; }
.summary ul { list-style: none; margin: 0; padding: 0; }
.summary li { margin: 4px 0; font-size: 22px; line-height: 1.4; overflow-wrap: anywhere; }
footer { margin-top: 24px; text-align: center; }
.note { margin: 0 0 12px; font-size: 20px; line-height: 1.4; color: #5E6670; }
.source { margin: 0 0 12px; font-size: 20px; line-height: 1.35; font-weight: 600; color: #536074; }
.social { display: flex; justify-content: center; flex-wrap: wrap; gap: 14px 64px; font-size: 22px; font-weight: 700; }
.social-item { display: inline-flex; align-items: center; gap: 12px; }
.social-icon { width: 28px; height: 28px; flex: 0 0 28px; }
.game-state th, .game-state td { padding: 7px 10px; }
.game-state .team-name { font-size: 26px; }
.state-cell { display: grid; grid-template-columns: minmax(0, 1fr) 62px; align-items: center; gap: 12px; }
.state-track { width: 100%; min-width: 0; height: 24px; }
.state-bar { height: 100%; }
.state-value { font-size: 26px; font-weight: 700; text-align: left; font-variant-numeric: tabular-nums; }
[data-text] { word-wrap: break-word; overflow-wrap: anywhere; }
@media (max-width: 1100px) {
  .report { padding: 20px; }
  .contributors { gap: 6px; }
  .player { flex-direction: column; gap: 4px; text-align: center; }
  .player-photo, .player-fallback { flex-basis: auto; }
  .team { flex-wrap: wrap; }
  th { font-size: 24px; }
  .state-cell { grid-template-columns: minmax(0, 1fr); gap: 5px; }
  .summaries { gap: 14px; }
}
@media (max-width: 650px) {
  .brand { flex-wrap: wrap; justify-content: center; gap: 12px; }
  .brand-logo { width: min(320px, 76vw); }
  .contributors { grid-template-columns: minmax(0, 1fr); gap: 12px; }
  .summaries { grid-template-columns: minmax(0, 1fr); }
  .team-badge { width: 28px; height: 34px; flex-basis: 28px; }
  .legend { justify-content: center; }
}
'''


def _html_image_uri(image) -> str | None:
    if image is None:
        return None
    buffer = io.BytesIO()
    if isinstance(image, (str, Path)):
        path = Path(image)
        if not path.is_file():
            return None
        with Image.open(path) as opened:
            opened.convert("RGBA").save(buffer, format="PNG")
    else:
        Image.fromarray(np.asarray(image).astype(np.uint8)).convert("RGBA").save(buffer, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buffer.getvalue()).decode("ascii")


def _html_fonts() -> str:
    css = []
    for key, family, weight in (("title", "BebasNeue", 400), ("reg", "Poppins", 400),
                                ("semi", "Poppins", 600), ("bold", "Poppins", 700)):
        properties = FONTS.get(key)
        path = Path(properties.get_file()) if properties and properties.get_file() else None
        if path and path.is_file():
            encoded = base64.b64encode(path.read_bytes()).decode("ascii")
            css.append(f"@font-face{{font-family:'{family}';font-weight:{weight};"
                       f"src:url(data:font/ttf;base64,{encoded}) format('truetype');font-display:block;}}")
    return "\n".join(css)


def _html_text(value) -> str:
    return html_escape(str(value), quote=True)


def _html_image(image, class_name: str, label: str, fallback: str = "") -> str:
    uri = _html_image_uri(image)
    if uri:
        return f'<img class="{class_name}" src="{uri}" alt="{_html_text(label)}">'
    return f'<span class="{class_name.replace("photo", "fallback").replace("badge", "fallback")}">{_html_text(fallback)}</span>'


def _html_summaries(blocks: list[tuple]) -> str:
    sections = []
    for heading, colour, lines in blocks:
        items = "".join(f'<li data-text>{_html_text(line)}</li>' for line in lines)
        sections.append(f'<section class="summary"><h2 data-text style="color:{colour}">'
                        f'{_html_text(heading)}</h2><ul>{items}</ul></section>')
    return '<div class="summaries">' + "".join(sections) + '</div>'


def _html_document(title: str, subtitle: str, table: str, blocks: list[tuple], footer: str,
                   note: str | None, channel: str, brand_logo, legend: str = "") -> str:
    logo = _html_image(brand_logo, "brand-logo", channel, "◆") if brand_logo else '<span class="brand-fallback">◆</span>'
    x_icon = '<svg class="social-icon" viewBox="0 0 24 24" aria-hidden="true"><path fill="#111827" d="M18.9 2H22l-6.8 7.8L23.2 22h-6.3l-4.9-6.5L6.3 22H3.2l7.3-8.4L.8 2h6.5l4.4 5.8L18.9 2zm-1.1 18h1.7L6.4 3.9H4.6L17.8 20z"/></svg>'
    yt_icon = '<svg class="social-icon" viewBox="0 0 28 22" aria-hidden="true"><rect width="28" height="22" rx="4" fill="#ED1D24"/><path d="M11 6L20 11L11 16z" fill="white"/></svg>'
    return ('<!doctype html><html lang="en"><head><meta charset="utf-8">'
            '<meta name="viewport" content="width=device-width,initial-scale=1">'
            f'<title>{_html_text(title)}</title><style>{_html_fonts()}\n{REPORT_CSS}</style></head><body>'
            f'<main class="report"><div class="brand">{logo}<span class="brand-name" data-text>{_html_text(channel)}</span></div>'
            f'<h1 data-text>{_html_text(title.upper())}</h1><p class="subtitle" data-text>{_html_text(subtitle)}</p>'
            f'{table}{legend}{_html_summaries(blocks)}<footer>'
            f'<p class="note" data-text>{_html_text(note or "")}</p><p class="source" data-text>{_html_text(footer)}</p>'
            f'<div class="social"><span class="social-item">{x_icon}<span data-text>{_html_text(X_HANDLE)}</span></span>'
            f'<span class="social-item">{yt_icon}<span data-text>{_html_text(YOUTUBE_HANDLE)}</span></span></div>'
            '</footer></main></body></html>')


def _export_html_png(document: str, out: str, subtitle: str) -> None:
    output = Path(out)
    output.parent.mkdir(parents=True, exist_ok=True)
    html_path = output.with_suffix(".html")
    html_path.write_text(document, encoding="utf-8")
    def render() -> dict:
        with sync_playwright() as playwright:
            launch_options: dict[str, Any] = {"headless": True}
            executable = shutil.which("chromium") or shutil.which("chromium-browser") or shutil.which("google-chrome")
            if executable:
                launch_options["executable_path"] = executable
            try:
                browser = playwright.chromium.launch(**launch_options)
            except Exception as error:
                if not any(message in str(error) for message in ("Executable doesn't exist", "Host system is missing dependencies", "error while loading shared libraries")):
                    raise RuntimeError(f"Chromium could not start: {error}") from error
                command = [sys.executable, "-m", "playwright", "install", "--with-deps", "chromium", "--only-shell"]
                subprocess.run(command, check=True)
                browser = playwright.chromium.launch(**launch_options)
            try:
                context = browser.new_context(viewport={"width": HTML_DESIGN_WIDTH, "height": 1000},
                                              device_scale_factor=PNG_EXPORT_WIDTH / HTML_DESIGN_WIDTH)
                page = context.new_page()
                page.set_content(document, wait_until="load")
                page.evaluate("""async () => {
                    await document.fonts.ready;
                    await Promise.all(Array.from(document.images).map(image => image.decode()));
                }""")
                audit = page.evaluate("""() => {
                    const report = document.querySelector('.report');
                    const canvas = report.getBoundingClientRect();
                    const boxes = [];
                    const issues = [];
                    for (const element of document.querySelectorAll('[data-text]')) {
                        const range = document.createRange();
                        range.selectNodeContents(element);
                        const text = element.textContent.trim();
                        const parent = element.closest('td,th,.summary,footer,.brand') || report;
                        const allowed = parent.getBoundingClientRect();
                        for (const rect of range.getClientRects()) {
                            if (!text || rect.width === 0 || rect.height === 0) continue;
                            if (rect.left < canvas.left - 1 || rect.right > canvas.right + 1 ||
                                rect.top < canvas.top - 1 || rect.bottom > canvas.bottom + 1 ||
                                rect.left < allowed.left - 1 || rect.right > allowed.right + 1 ||
                                rect.top < allowed.top - 1 || rect.bottom > allowed.bottom + 1) {
                                issues.push('Out of bounds: ' + text);
                            }
                            boxes.push({text, left:rect.left, right:rect.right, top:rect.top, bottom:rect.bottom, element});
                        }
                    }
                    for (let first = 0; first < boxes.length; first++) {
                        for (let second = first + 1; second < boxes.length; second++) {
                            const a = boxes[first], b = boxes[second];
                            if (a.element === b.element) continue;
                            const overlapWidth = Math.min(a.right,b.right)-Math.max(a.left,b.left);
                            const overlapHeight = Math.min(a.bottom,b.bottom)-Math.max(a.top,b.top);
                            if (overlapWidth > 1 && overlapHeight > 1) issues.push('Text overlap: '+a.text+' / '+b.text);
                        }
                    }
                    for (const image of document.querySelectorAll('img')) {
                        const rect = image.getBoundingClientRect();
                        if (rect.left < canvas.left - 1 || rect.right > canvas.right + 1 ||
                            rect.top < canvas.top - 1 || rect.bottom > canvas.bottom + 1) issues.push('Image outside canvas: '+image.alt);
                        for (const box of boxes) {
                            if (Math.min(rect.right,box.right)-Math.max(rect.left,box.left) > 1 &&
                                Math.min(rect.bottom,box.bottom)-Math.max(rect.top,box.top) > 1) issues.push('Image/text overlap: '+image.alt+' / '+box.text);
                        }
                    }
                    if (document.documentElement.scrollWidth > window.innerWidth + 1) issues.push('Horizontal overflow');
                    return {width:canvas.width,height:canvas.height,rows:document.querySelectorAll('tbody tr').length,
                            issues:[...new Set(issues)],
                            fonts:Object.fromEntries(['h1','.subtitle','th','.team-name','.player-name','.note','.source']
                                .map(selector=>[selector,document.querySelector(selector)?getComputedStyle(document.querySelector(selector)).fontSize:null]))};
                }""")
                if audit["issues"]:
                    raise ValueError("PNG layout validation failed: " + "; ".join(audit["issues"][:8]))
                if audit["height"] * PNG_EXPORT_WIDTH / HTML_DESIGN_WIDTH > 30000:
                    raise ValueError("The report is too tall for one PNG; inspect unusually long input text")
                png_bytes = page.screenshot(full_page=True, type="png", animations="disabled")
            finally:
                browser.close()
        metadata = PngInfo()
        metadata.add_text("Description", subtitle)
        with Image.open(io.BytesIO(png_bytes)) as image:
            if image.width != PNG_EXPORT_WIDTH:
                raise ValueError(f"Unexpected PNG width {image.width}; expected {PNG_EXPORT_WIDTH}")
            temporary = output.with_suffix(".pending.png")
            image.save(temporary, format="PNG", pnginfo=metadata)
        temporary.replace(output)
        return audit
    # Colab runs an asyncio loop; the browser's synchronous API uses its own worker.
    with ThreadPoolExecutor(max_workers=1) as executor:
        audit = executor.submit(render).result()
    output.with_suffix(".layout.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
    print(f"Saved {output}: {audit['rows']} rows; layout bounds and overlap checks passed.")



def _draw_ranked_chart(shares, cats, title, subtitle, footer, out, logos=None, short=None,
                       channel=CHANNEL, brand_logo=None, note=None, top_players=None, photos=None,
                       mode="xg", layout="1", rank_offset=0):
    logos, short = logos or {}, short or {}
    top_players, photos = top_players or {}, photos or {}
    is_xg = mode == "xg" and {"xg_total", "goals", "diff"}.issubset(shares.columns)
    is_defcon = mode == "defcon"
    role_colours = {str(category): colour for category, _, colour in cats}
    widths = [3,19,16,6,7,7,42] if is_xg else [3,19,18,10,6,44] if is_defcon else [3,20,32,45]
    headers = ["#", "TEAM", "MIX (%)"]
    headers += ["xG", "GOALS", "G−xG"] if is_xg else ["ACTIONS", "/90"] if is_defcon else []
    headers += ["TOP 3 xG CONTRIBUTORS" if is_xg else "TOP 3 DEFCON CONTRIBUTORS" if is_defcon else "TOP 3 CONTRIBUTORS"]
    rows = []
    for index, (_, record) in enumerate(shares.iterrows()):
        team = str(record["team"])
        badge = _html_image(logos.get(team), "team-badge", team, short.get(team, team[:3]))
        cells = [f'<td class="rank"><span data-text>{rank_offset + index + 1}</span></td>',
                 f'<td><div class="team">{badge}<span class="team-name" data-text>{_html_text(team)}</span></div></td>']
        segments, percentages = [], []
        for category, label, colour in cats:
            percentage = float(record[category])
            if not math.isfinite(percentage) or not -0.01 <= percentage <= 100.01:
                raise ValueError(f"Invalid positional percentage for {team}")
            segments.append(f'<span class="mix-segment" style="width:{max(0,percentage):.6f}%;background:{colour}"></span>')
            percentages.append(f'<span class="mix-pct" data-text title="{_html_text(label)}" style="color:{colour}">{percentage:.0f}%</span>')
        cells.append(f'<td><div class="mix-track">{"".join(segments)}</div>'
                     f'<div class="mix-labels" style="--category-count:{len(cats)}">{"".join(percentages)}</div></td>')
        if is_xg:
            difference_class = "metric-positive" if record["diff"] >= 0 else "metric-negative"
            cells += [f'<td class="metric"><span data-text>{record["xg_total"]:.1f}</span></td>',
                      f'<td class="metric"><span data-text>{int(round(record["goals"]))}</span></td>',
                      f'<td class="metric {difference_class}"><span data-text>{record["diff"]:+.1f}</span></td>']
        elif is_defcon:
            cells += [f'<td class="metric"><span data-text>{int(round(record["actions"]))}</span></td>',
                      f'<td class="metric"><span data-text>{record["actions90"]:.1f}</span></td>']
        contributors = []
        for player in top_players.get(team, [])[:3]:
            name = str(player.get("name") or "Unknown")
            image_key = str(player.get("photo") or player.get("code"))
            initials = "".join(part[0] for part in name.split() if part)[:2]
            picture = _html_image(crop_headshot(photos.get(image_key)), "player-photo", name, initials)
            value = float(player.get("value", 0))
            value_text = f"{value:.1f} xG" if is_xg else f"{value:.0f} actions"
            colour = role_colours.get(str(player.get("role", "")), "#42526A")
            contributors.append(f'<div class="player">{picture}<div class="player-copy">'
                                f'<div class="player-name" data-text>{_html_text(name)}</div>'
                                f'<div class="player-value" data-text style="color:{colour}">{value_text}</div></div></div>')
        cells.append('<td><div class="contributors">' + "".join(contributors) + '</div></td>')
        rows.append('<tr>' + "".join(cells) + '</tr>')
    column_group = '<colgroup>' + "".join(f'<col style="width:{width}%">' for width in widths) + '</colgroup>'
    table = ('<table>' + column_group + '<thead><tr>' + "".join(f'<th scope="col" data-text>{_html_text(header)}</th>' for header in headers)
             + '</tr></thead><tbody>' + "".join(rows) + '</tbody></table>')
    blocks = []
    if is_xg:
        blocks = [
            ("RELIANT ON ONE PLAYER", GOOD, [f"{row.team}: {row.top_player} ({row.top_share:.0f}%)" for row in shares.nlargest(3,"top_share").itertuples()]),
            ("BEATING xG", GOOD, [f"{row.team}: {row.diff:+.1f} G−xG" for row in shares.nlargest(3,"diff").itertuples()]),
            ("BELOW xG", BAD, [f"{row.team}: {row.diff:+.1f} G−xG" for row in shares.nsmallest(3,"diff").itertuples()]),
        ]
    elif is_defcon:
        players = sorted([dict(player,team=team) for team, values in top_players.items() for player in values],
                         key=lambda player: player["value"], reverse=True)[:3]
        blocks = [
            ("MOST DEFCON ACTIONS", GOOD, [f"{row.team}: {int(row.actions)}" for row in shares.nlargest(3,"actions").itertuples()]),
            ("ACTIONS PER 90", "#3B82F6", [f"{row.team}: {row.actions90:.1f}" for row in shares.nlargest(3,"actions90").itertuples()]),
            ("TOP PLAYERS", "#8E68D6", [f"{player['name']} ({player['team']}): {int(player['value'])}" for player in players]),
        ]
    legend = '<div class="legend">' + "".join(
        f'<span class="legend-item"><span class="swatch" style="background:{colour}"></span><span data-text>{_html_text(label)}</span></span>'
        for _, label, colour in cats) + '</div>'
    document = _html_document(title, subtitle, table, blocks, footer, note, channel, brand_logo, legend)
    _export_html_png(document, out, subtitle)


def generate_xg_bundle(args):
    root_dir = resolve_out_dir(args.out_dir)
    out_dir = root_dir / "xG"
    asset_dir = root_dir / "_assets"
    clear_report_folder(out_dir)

    boot = fpl_bootstrap()
    period = reporting_period(boot, fpl_fixtures())
    gw = period["gameweek"]
    if gw == 0:
        print("No finished gameweek yet - nothing to chart.")
        return None

    period_boot = boot
    if args.source == "fpl":
        period_boot = cumulative_player_bootstrap(
            boot, period, ("minutes", "expected_goals", "goals_scored"), asset_dir / "fpl_event_data")
        df, cats = mix_fpl_position(period_boot)
        title, what, src = ("Who Creates Each Team's xG", "Cumulative team xG share by player position", "FPL API")
    elif args.source == "understat":
        df, cats = mix_understat(boot)
        title, what, src = ("How Teams Create xG", "Cumulative share of total xG by situation", "Understat")
    else:
        if not args.csv:
            raise SystemExit("--source csv needs --csv FILE")
        df, cats = pd.read_csv(args.csv), CATS_SITUATION
        title, what, src = ("How Teams Create xG", "Cumulative share of total xG by situation", args.csv_source)

    shares, cats = prepare(df, cats, args.sort)
    if len(shares) != len(boot["teams"]) or shares["team"].nunique() != len(boot["teams"]):
        raise ValueError("xG must contain every Premier League club exactly once")

    team_info = teams_table(boot)
    short = dict(zip(team_info["name"], team_info["short_name"]))
    logos = {} if args.no_logos else load_logos(boot, asset_dir / "badges")
    players = top_three_by_team(period_boot, "expected_goals")
    photos = {} if args.no_photos else load_player_photos(players, asset_dir / "player_photos")
    load_fonts(asset_dir / "fonts")
    brand_logo = resolve_logo(args.logo, asset_dir)

    y0 = season_start_year(boot)
    display_range = period["range"] if args.source == "fpl" else "Source season totals; cutoff unverified"
    subtitle_base = f"{what} | Premier League {period['season']} | {display_range}"
    if args.source == "fpl" and period["provisional"]:
        subtitle_base += " | Provisional"
    footer = f"{args.channel} | {src} | {display_range} | {dt.date.today():%d.%m.%Y}"
    note = "Expected goals measure chance quality. xG and goals are credited to each player's current club."

    outputs = _output_paths(out_dir)
    texts = _text_paths(out_dir)

    _draw_ranked_chart(shares, cats, title, subtitle_base, footer, str(outputs['1']), logos, short,
                       args.channel, brand_logo, note, players, photos, mode="xg", layout="1", rank_offset=0)
    if args.source == "fpl":
        caption = _xg_post_for_section(shares, gw, args.channel, "All 20 teams", period["season"], period["provisional"])
    else:
        caption = _fit_x_caption([
            f"{args.channel} | Premier League xG {period['season']}\n"
            f"{src} season totals; the source's gameweek cutoff is unverified.\n"
            "Expected goals describe chance quality for Fantasy Premier League.\n#FPL #ExpectedGoals #PremierLeague"
        ])
    _save_text(texts['1'], caption)
    _save_report_metadata(out_dir / "1.metadata.json", period, {
        "report": "xG", "team_count": len(shares), "source": src,
        "source_cutoff_verified": args.source == "fpl",
        "total_xg": float(shares["xg_total"].sum()) if "xg_total" in shares else None,
        "total_goals": float(shares["goals"].sum()) if "goals" in shares else None,
    })
    clear_report_folder(out_dir)
    print(f"xG: {display_range}, {period['fixture_count']} FPL fixtures; {period['status']}.")
    if in_colab():
        display(IPImage(str(outputs['1'])))
    return outputs


# Override notebook entrypoint

def run(args):
    return generate_xg_bundle(args)


def main(argv=None):
    return run(parse(argv))


## 2 · xG (one 20-team page)
Creates `xG/1.png` and `xG/1.txt`.


In [3]:
xg_outputs = main()
xg_outputs

Output hidden; open in https://colab.research.google.com to view.

## 3 · DEFCON (one 20-team page)
Creates `DEFCON/1.png` and `DEFCON/1.txt`.

**Data note:** FPL fixture DEFCON totals can disagree with gameweek live data. This notebook reports the discrepancy and uses validated gameweek live player data; goals and missing-match checks still fail on inconsistency.


In [ ]:
# DEFCON · one full-page 20-team report

def defcon_team_data(boot):
    players = pd.DataFrame(boot["elements"]).copy()
    if "element_type" not in players or "minutes" not in players:
        raise ValueError("Required FPL player fields are missing")
    outfield = players[players["element_type"].isin((2, 3, 4))].copy()
    explicit = pd.to_numeric(outfield.get("defensive_contribution", pd.Series(dtype=float)), errors="coerce")
    if len(explicit) == len(outfield) and explicit.notna().any():
        outfield["defcon_actions"] = explicit.fillna(0)
        provenance = "FPL defensive_contribution season actions"
    else:
        needed = {"clearances_blocks_interceptions", "tackles", "recoveries"}
        if not needed.issubset(outfield.columns):
            raise ValueError("No FPL DEFCON action data in bootstrap. Check API fields before publishing.")
        cbi = pd.to_numeric(outfield["clearances_blocks_interceptions"], errors="coerce").fillna(0)
        tackles = pd.to_numeric(outfield["tackles"], errors="coerce").fillna(0)
        recoveries = pd.to_numeric(outfield["recoveries"], errors="coerce").fillna(0)
        outfield["defcon_actions"] = cbi + tackles + np.where(outfield["element_type"].isin([3, 4]), recoveries, 0)
        provenance = "FPL CBI + tackles (+ recoveries for MID/FWD)"
    outfield["mins"] = pd.to_numeric(outfield["minutes"], errors="coerce").fillna(0)
    position_names = {2: "defenders", 3: "midfielders", 4: "forwards"}
    outfield["category"] = outfield["element_type"].map(position_names)
    totals = outfield.pivot_table(index="team", columns="category", values="defcon_actions", aggfunc="sum", fill_value=0)
    for position in position_names.values():
        if position not in totals:
            totals[position] = 0.0
    totals = totals[list(position_names.values())].copy()
    totals["actions"] = totals.sum(axis=1)
    totals["minutes"] = outfield.groupby("team")["mins"].sum()
    totals["actions90"] = (totals["actions"] * 90 / totals["minutes"].replace(0, np.nan)).fillna(0)
    names = teams_table(boot).set_index("id")["name"]
    totals.insert(0, "team", names.reindex(totals.index).values)
    team_players = {}
    sorted_players = outfield.sort_values(["defcon_actions", "id"], ascending=[False, True])
    for team_id, group in sorted_players.groupby("team", sort=False):
        team_players[names.loc[team_id]] = [
            {"name": str(rec.web_name), "value": float(rec.defcon_actions),
             "code": str(getattr(rec, "code", "")), "photo": str(getattr(rec, "photo", "")),
             "role": position_names[int(rec.element_type)]}
            for rec in group.head(3).itertuples(index=False)
        ]
    columns = [("defenders", "DEF", "#27AE60"), ("midfielders", "MID", "#3B82F6"), ("forwards", "FWD", "#F28C38")]
    return totals.reset_index(drop=True), columns, team_players, provenance


def defcon_main(argv=None):
    args = parse(argv)
    root_dir = resolve_out_dir(args.out_dir)
    out_dir = root_dir / "DEFCON"
    asset_dir = root_dir / "_assets"
    clear_report_folder(out_dir)

    boot = fpl_bootstrap()
    period = reporting_period(boot, fpl_fixtures())
    gw = period["gameweek"]
    if gw == 0:
        print("No completed gameweek; DEFCON not generated.")
        return None

    period_boot = cumulative_player_bootstrap(
        boot, period, ("minutes", "defensive_contribution", "goals_scored"), asset_dir / "fpl_event_data")
    stats, categories, top_players, provenance = defcon_team_data(period_boot)
    shares, cats = prepare(stats, categories, sort="share")
    shares = shares.sort_values("actions", ascending=False).reset_index(drop=True)
    if len(shares) != len(boot["teams"]) or shares["team"].nunique() != len(boot["teams"]):
        raise ValueError("DEFCON must contain every Premier League club exactly once")

    team_info = teams_table(boot)
    short = dict(zip(team_info["name"], team_info["short_name"]))
    badges = {} if args.no_logos else load_logos(boot, asset_dir / "badges")
    pictures = {} if args.no_photos else load_player_photos(top_players, asset_dir / "player_photos")
    load_fonts(asset_dir / "fonts")
    logo = resolve_logo(args.logo, asset_dir)

    year = season_start_year(boot)
    season = period["season"]
    subtitle_base = f"Cumulative team defensive actions by position | Premier League {season} | GW1–GW{gw}"
    footer = f"FPL action data | {period['range']} | {period['status']} | {dt.date.today():%d.%m.%Y}"
    subtitle_base += " | Provisional" if period["provisional"] else ""
    note = "DEFCON actions, not FPL points. /90 uses summed outfield player-minutes; actions credited to current clubs."

    outputs = _output_paths(out_dir)
    texts = _text_paths(out_dir)

    _draw_ranked_chart(shares, cats, "THE DEFCON REPORT", subtitle_base, footer, str(outputs['1']),
                       badges, short, args.channel, logo, note, top_players, pictures,
                       mode="defcon", layout="1", rank_offset=0)
    _save_text(texts['1'], _defcon_post_for_section(
        shares, gw, args.channel, "All 20 teams", season, top_players, period["provisional"]))
    _save_report_metadata(out_dir / "1.metadata.json", period, {
        "report": "DEFCON", "total_actions": float(shares["actions"].sum()),
        "total_player_minutes": float(shares["minutes"].sum()),
        "team_count": len(shares), "method": provenance,
    })
    clear_report_folder(out_dir)
    print(f"DEFCON: {period['range']}, {period['fixture_count']} fixtures; {period['status']}.")
    if in_colab():
        display(IPImage(str(outputs['1'])))
    return outputs


defcon_outputs = defcon_main()
defcon_outputs


## 4 · GAME_STATE (FPL validated FotMob goals)
Creates `GAME_STATE/1.png` and `GAME_STATE/1.txt` only after full fixture validation. Retains last verified outputs if incomplete.


In [ ]:
# GAME_STATE: FPL-verified, FotMob-derived goal timelines with permanent local event archive.
import datetime as dt
import json
import re
from pathlib import Path
from typing import Any

FOTMOB_API = "https://www.fotmob.com/api/data"
FOTMOB_LEAGUE_ID = 47
FOTMOB_TEAM_ALIASES = {
    "Manchester City": "Man City", "Manchester United": "Man Utd",
    "Newcastle United": "Newcastle", "Wolverhampton Wanderers": "Wolves",
    "Tottenham Hotspur": "Spurs", "Tottenham": "Spurs",
    "Nottingham Forest": "Nott'm Forest", "Brighton and Hove Albion": "Brighton",
    "Brighton & Hove Albion": "Brighton", "West Ham United": "West Ham",
    "AFC Bournemouth": "Bournemouth", "Leeds United": "Leeds",
    "Sunderland AFC": "Sunderland", "Leicester City": "Leicester",
    "Crystal Palace FC": "Crystal Palace", "Fulham FC": "Fulham",
    "Coventry City": "Coventry", "Ipswich Town": "Ipswich",
}


def _fotmob_team_normalize(value: str) -> str:
    return re.sub(r"[^a-z0-9]", "", str(value).lower())


def _fotmob_name_map(boot: dict) -> dict[str, str]:
    official = teams_table(boot)["name"].tolist()
    name_map = {_fotmob_team_normalize(name): name for name in official}
    for alias, canonical in FOTMOB_TEAM_ALIASES.items():
        if canonical in official:
            name_map[_fotmob_team_normalize(alias)] = canonical
    return name_map


def _fotmob_team_name(raw: Any, mapping: dict[str, str]) -> str:
    if isinstance(raw, dict):
        raw = raw.get("name") or raw.get("longName") or raw.get("shortName")
    name = mapping.get(_fotmob_team_normalize(str(raw)))
    if not name:
        raise ValueError(f"Unknown FotMob team {raw!r}; revise FOTMOB_TEAM_ALIASES")
    return name


def _fotmob_fetch(path: str, params: dict[str, str]) -> dict:
    response = requests.get(FOTMOB_API + path, params=params,
                            headers=HEADERS, timeout=30)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError(f"Invalid FotMob JSON for {path}")
    return payload


def _fotmob_utc(value: str) -> dt.datetime:
    if not value:
        raise ValueError("Missing UTC kickoff")
    parsed = dt.datetime.fromisoformat(str(value).replace("Z", "+00:00"))
    if parsed.tzinfo is None:
        raise ValueError("Kickoff must contain a timezone")
    return parsed.astimezone(dt.timezone.utc)


def _finished_fpl_fixtures(boot: dict, gw: int, fixtures: list[dict] | None = None) -> list[dict]:
    fixtures = fixtures if fixtures is not None else fpl_fixtures()
    selected = [fixture for fixture in fixtures if fixture.get("event") is not None
                and 1 <= int(fixture["event"]) <= gw]
    if not selected or any(not fixture_has_ended(fixture) for fixture in selected):
        raise ValueError(f"GW1–GW{gw} still contains unfinished fixtures")
    return selected


def _fotmob_day_matches(date: dt.date, day_cache: dict[str, list[dict]]) -> list[dict]:
    key = date.strftime("%Y%m%d")
    if key not in day_cache:
        payload = _fotmob_fetch("/matches", {"date": key})
        result = []
        for league in payload.get("leagues", []):
            if not isinstance(league, dict):
                continue
            if str(league.get("id", league.get("primaryId", ""))) != str(FOTMOB_LEAGUE_ID):
                continue
            result.extend(item for item in league.get("matches", []) if isinstance(item, dict))
        day_cache[key] = result
    return day_cache[key]


def _fotmob_find_match(fixture: dict, home: str, away: str,
                       expected: tuple[int, int], names: dict[str, str],
                       day_cache: dict[str, list[dict]]) -> str:
    kickoff = _fotmob_utc(fixture["kickoff_time"])
    possible: dict[str, dict] = {}
    for shift in (-1, 0, 1):
        for match in _fotmob_day_matches((kickoff + dt.timedelta(days=shift)).date(), day_cache):
            if not match.get("id"):
                continue
            try:
                if (_fotmob_team_name(match.get("home", {}), names),
                    _fotmob_team_name(match.get("away", {}), names)) != (home, away):
                    continue
                if (int(match["home"]["score"]), int(match["away"]["score"])) != expected:
                    continue
                if not match.get("status", {}).get("finished"):
                    continue
                possible[str(match["id"])] = match
            except (ValueError, TypeError, KeyError):
                continue
    if len(possible) != 1:
        raise ValueError(f"FotMob fixture lookup matched {len(possible)} entries for {home} vs {away}")
    return next(iter(possible))


def _fotmob_goal_minute(event: dict) -> float:
    raw = event.get("timeStr", event.get("time"))
    if raw is None:
        raw = event.get("time")
    if isinstance(raw, str):
        stripped = raw.strip().replace("’", "'").replace("′", "'").rstrip("'")
        found = re.fullmatch(r"(\d+)(?:\s*\+\s*(\d+))?", stripped)
        if not found:
            raise ValueError(f"Unknown FotMob goal time format: {raw!r}")
        # Added-time goals belong to their half's regulation boundary.
        minute = float(int(found.group(1)))
    elif isinstance(raw, (int, float)):
        minute = float(raw)
    else:
        raise ValueError(f"Unknown FotMob goal minute: {raw!r}")
    if not np.isfinite(minute) or minute < 0:
        raise ValueError("Negative or invalid goal minute")
    return min(90.0, minute)


def _fotmob_parse_verified_goals(details: dict, expected: tuple[int, int]) -> list[dict]:
    events_container = details.get("content", {}).get("matchFacts", {}).get("events", {})
    events = events_container.get("events", []) if isinstance(events_container, dict) else []
    if not isinstance(events, list):
        raise ValueError("FotMob goals are not an event list")
    goals = [event for event in events if isinstance(event, dict)
             and str(event.get("type", "")).lower() in ("goal", "owngoal", "own goal")
             and event.get("isPenaltyShootoutEvent") is not True]
    if len(goals) != sum(expected):
        raise ValueError(f"FotMob has {len(goals)} events, but FPL score needs {sum(expected)} goals")
    # FotMob supplies the score BEFORE each goal; this also solves own-goal attribution.
    if any(event.get("homeScore") is None or event.get("awayScore") is None for event in goals):
        raise ValueError("FotMob does not provide pre-goal scores for all goals")
    goals = sorted(goals, key=lambda event: int(event["homeScore"]) + int(event["awayScore"]))
    verified = []
    current = (0, 0)
    for index, goal in enumerate(goals):
        previous = (int(goal["homeScore"]), int(goal["awayScore"]))
        if previous != current:
            raise ValueError(f"FotMob inconsistent goal score sequence at event {index}: {previous} vs {current}")
        following = (int(goals[index+1]["homeScore"]), int(goals[index+1]["awayScore"])) if index+1 < len(goals) else expected
        delta = (following[0] - current[0], following[1] - current[1])
        if delta not in ((1, 0), (0, 1)):
            raise ValueError(f"Invalid FotMob score transition: {current} -> {following}")
        side = "h" if delta == (1, 0) else "a"
        verified.append({"minute": _fotmob_goal_minute(goal), "side": side,
                         "own_goal": bool(goal.get("ownGoal")),
                         "event_id": str(goal.get("eventId", "")),
                         "source_time": str(goal.get("timeStr", goal.get("time", "")))})
        current = following
    if current != expected:
        raise ValueError("FotMob goals do not reconcile to official FPL score")
    if any(verified[i]["minute"] > verified[i+1]["minute"] for i in range(len(verified)-1)):
        raise ValueError("FotMob chronological goal minutes contradict score progression")
    return verified


def _timeline_from_verified_goals(goals: list[dict], expected: tuple[int, int]) -> dict[str, float]:
    if (sum(goal["side"] == "h" for goal in goals),
        sum(goal["side"] == "a" for goal in goals)) != expected:
        raise ValueError("Archived goal totals do not match FPL score")
    totals = {"winning": 0.0, "drawing": 0.0, "losing": 0.0}
    home_score = away_score = 0
    previous = 0.0
    for goal in goals:
        minute = float(goal["minute"])
        if not np.isfinite(minute) or not previous <= minute <= 90:
            raise ValueError("Archived goals are not chronological within regulation")
        state = "winning" if home_score > away_score else "losing" if home_score < away_score else "drawing"
        totals[state] += minute - previous
        if goal["side"] == "h":
            home_score += 1
        elif goal["side"] == "a":
            away_score += 1
        else:
            raise ValueError("Invalid archived scoring side")
        previous = minute
    state = "winning" if home_score > away_score else "losing" if home_score < away_score else "drawing"
    totals[state] += 90.0 - previous
    if abs(sum(totals.values()) - 90.0) > 1e-7:
        raise ValueError("Goal states did not total 90 minutes")
    return totals


def _verified_fotmob_fixture(fixture: dict, home: str, away: str, expected: tuple[int, int],
                              names: dict[str, str], day_cache: dict[str, list[dict]],
                              archive: Path) -> dict[str, float]:
    archive.mkdir(parents=True, exist_ok=True)
    fixture_id = str(fixture["id"])
    cache_path = archive / f"fpl_{fixture_id}.json"
    def matches_current(item: dict) -> bool:
        return (item.get("fixture_id") == fixture_id and item.get("home") == home
                and item.get("away") == away and tuple(item.get("score", [])) == expected
                and item.get("kickoff") == fixture.get("kickoff_time")
                and item.get("provider") == "FotMob")
    cached = {}
    if cache_path.exists():
        try:
            cached = json.loads(cache_path.read_text(encoding="utf-8"))
        except (ValueError, OSError):
            cached = {}
        if matches_current(cached) and cached.get("schema_version") == 2:
            return _timeline_from_verified_goals(cached["goals"], expected)
    match_id = (str(cached["fotmob_match_id"]) if matches_current(cached) and cached.get("fotmob_match_id")
                else _fotmob_find_match(fixture, home, away, expected, names, day_cache))
    details = _fotmob_fetch("/matchDetails", {"matchId": str(match_id)})
    general = details.get("general", {})
    header = details.get("header", {})
    if str(general.get("matchId")) != str(match_id) or int(general.get("leagueId", 0)) != FOTMOB_LEAGUE_ID:
        raise ValueError("FotMob match identity/league mismatch")
    if not general.get("finished") or not header.get("status", {}).get("finished"):
        raise ValueError("FotMob match is not confirmed finished")
    if (_fotmob_team_name(general.get("homeTeam", {}), names),
        _fotmob_team_name(general.get("awayTeam", {}), names)) != (home, away):
        raise ValueError("FotMob teams differ from FPL fixture")
    if abs((_fotmob_utc(general["matchTimeUTCDate"]) - _fotmob_utc(fixture["kickoff_time"])).total_seconds()) > 18*3600:
        raise ValueError("FotMob kickoff timestamp mismatch")
    header_teams = header.get("teams", [])
    if len(header_teams) != 2 or tuple(int(x["score"]) for x in header_teams) != expected:
        raise ValueError("FotMob final score differs from FPL")
    goals = _fotmob_parse_verified_goals(details, expected)
    minutes = _timeline_from_verified_goals(goals, expected)
    stored = {"schema_version": 2, "time_basis": "regulation_90", "provider": "FotMob", "fotmob_match_id": str(match_id),
              "fixture_id": fixture_id, "kickoff": fixture.get("kickoff_time"),
              "home": home, "away": away, "score": list(expected), "goals": goals}
    temp = cache_path.with_suffix(".tmp")
    temp.write_text(json.dumps(stored, ensure_ascii=False, indent=2), encoding="utf-8")
    temp.replace(cache_path)
    return minutes


def load_game_state_data(boot: dict, gw: int, cache_dir: Path, fixtures: list[dict] | None = None) -> pd.DataFrame:
    fixtures = _finished_fpl_fixtures(boot, gw, fixtures)
    name_map = _fotmob_name_map(boot)
    teams = teams_table(boot)
    names_by_id = dict(zip(teams["id"], teams["name"]))
    totals = {name: {"winning": 0.0, "drawing": 0.0, "losing": 0.0} for name in teams["name"]}
    archive = cache_dir / "verified_fotmob_events"
    scoreboard_cache: dict[str, list[dict]] = {}
    failures = []
    verified_count = 0
    for fixture in fixtures:
        home, away = names_by_id[int(fixture["team_h"])], names_by_id[int(fixture["team_a"])]
        expected = (int(fixture["team_h_score"]), int(fixture["team_a_score"]))
        try:
            minutes = _verified_fotmob_fixture(fixture, home, away, expected,
                                               name_map, scoreboard_cache, archive)
        except (ValueError, KeyError, TypeError, OSError, requests.RequestException) as error:
            failures.append(f"{home} vs {away}: {error}")
            continue
        for state, duration in minutes.items():
            totals[home][state] += duration
        totals[away]["winning"] += minutes["losing"]
        totals[away]["losing"] += minutes["winning"]
        totals[away]["drawing"] += minutes["drawing"]
        verified_count += 1
    if failures or verified_count != len(fixtures):
        print(f"GAME STATE: {verified_count}/{len(fixtures)} fixtures verified from FotMob/local archive.")
        for problem in failures[:20]:
            print(" -", problem)
        raise ValueError("Some goal timelines are unverified; existing GAME_STATE images are preserved. "
                         "No partial-season percentages are published.")
    rows = []
    for team, values in totals.items():
        duration = sum(values.values())
        if not duration:
            raise ValueError(f"No verified fixture minutes for {team}")
        rows.append({"team": team,
                     "winning": values["winning"]/duration*100,
                     "drawing": values["drawing"]/duration*100,
                     "losing": values["losing"]/duration*100,
                     "minutes": duration})
    matches_by_team = {name: 0 for name in teams["name"]}
    for fixture in fixtures:
        matches_by_team[names_by_id[int(fixture["team_h"])]] += 1
        matches_by_team[names_by_id[int(fixture["team_a"])]] += 1
    for row in rows:
        if abs(row["minutes"] - matches_by_team[row["team"]] * 90) > 1e-7:
            raise ValueError(f"Incorrect cumulative regulation minutes for {row['team']}")
    output = pd.DataFrame(rows).sort_values(["winning", "drawing", "losing"],
                                            ascending=[False, False, True]).reset_index(drop=True)
    if len(output) != len(boot["teams"]) or output["team"].nunique() != len(boot["teams"]) or not np.allclose(output[["winning","drawing","losing"]].sum(axis=1), 100):
        raise ValueError("GAME_STATE failed 20-team/100-percent validation")
    print(f"GAME STATE: verified {verified_count}/{len(fixtures)} fixtures. "
          f"Archived events under {archive}.")
    output.attrs["verified_fixture_count"] = verified_count
    output.attrs["matches_by_team"] = matches_by_team
    return output

def draw_game_state_chart(df, title, subtitle, footer, out, logos=None, short=None,
                          channel=CHANNEL, brand_logo=None, note=None, layout="1"):
    logos, short = logos or {}, short or {}
    colours = {"losing":"#E67B87", "drawing":"#F2B366", "winning":"#49B675"}
    rows = []
    for _, record in df.iterrows():
        team = str(record["team"])
        badge = _html_image(logos.get(team), "team-badge", team, short.get(team,team[:3]))
        cells = [f'<td><div class="team">{badge}<span class="team-name" data-text>{_html_text(team)}</span></div></td>']
        for key in ("losing", "drawing", "winning"):
            value = float(record[key])
            if not math.isfinite(value) or not 0 <= value <= 100:
                raise ValueError(f"Invalid {key} percentage for {team}")
            cells.append(f'<td><div class="state-cell"><div class="state-track"><div class="state-bar" '
                         f'style="width:{value:.6f}%;background:{colours[key]}"></div></div>'
                         f'<span class="state-value" data-text>{value:.0f}%</span></div></td>')
        rows.append('<tr>' + "".join(cells) + '</tr>')
    headers = ['TEAM','LOSING','DRAWING','WINNING']
    table = ('<table class="game-state"><colgroup><col style="width:25%"><col style="width:25%">'
             '<col style="width:25%"><col style="width:25%"></colgroup><thead><tr>'
             + "".join(f'<th scope="col" data-text>{heading}</th>' for heading in headers)
             + '</tr></thead><tbody>' + "".join(rows) + '</tbody></table>')
    blocks = [(f"MOST TIME {key.upper()}", colours[key],
               [f"{getattr(row,'team')}: {getattr(row,key):.0f}%" for row in df.nlargest(3,key).itertuples()])
              for key in ('winning','drawing','losing')]
    document = _html_document(title, subtitle, table, blocks, footer, note, channel, brand_logo)
    _export_html_png(document, out, subtitle)


def _game_state_post(df, gw, channel, label, season=None, provisional=False):
    top_win = df.nlargest(1, "winning").iloc[0]
    top_draw = df.nlargest(1, "drawing").iloc[0]
    top_lose = df.nlargest(1, "losing").iloc[0]
    status = " | provisional scores" if provisional else ""
    heading = f"{channel} | Game State {season or ''} | GW1–GW{gw}{status}".replace("  ", " ")
    insights = (f"Most time ahead: {top_win['team']} {top_win['winning']:.0f}%.\n"
                f"Level: {top_draw['team']} {top_draw['drawing']:.0f}%. "
                f"Behind: {top_lose['team']} {top_lose['losing']:.0f}%.")
    tags = "#FPL #PremierLeague #FPLStats"
    return _fit_x_caption([
        f"{heading}\n{insights}\nPremier League match-time shares, not win probabilities.\n{tags}",
        f"{heading}\n{insights}\n90-minute match-time shares, not win probabilities.\n{tags}",
        f"{heading}\n{insights}\nMatch-time shares, not win probabilities.\n#FPL #PremierLeague",
    ])


def game_state_main(argv=None):
    args = parse(argv)
    root_dir = resolve_out_dir(args.out_dir)
    out_dir = root_dir / "GAME_STATE"
    asset_dir = root_dir / "_assets"
    out_dir.mkdir(parents=True, exist_ok=True)
    boot = fpl_bootstrap()
    period = reporting_period(boot, fpl_fixtures())
    gw = period["gameweek"]
    if gw == 0:
        print("No finished gameweek; GAME_STATE not generated.")
        return None
    try:
        df = load_game_state_data(boot, gw, asset_dir / "gamestate_match_cache", period["fixtures"])
    except (ValueError, KeyError, TypeError, OSError, requests.RequestException) as error:
        print(f"GAME STATE not regenerated: {error}")
        print("Existing verified images are unchanged. A first-time run needs full FotMob access.")
        return _output_paths(out_dir) if (out_dir / "1.png").exists() else None
    team_info = teams_table(boot)
    short = dict(zip(team_info["name"], team_info["short_name"]))
    logos = {} if args.no_logos else load_logos(boot, asset_dir / "badges")
    load_fonts(asset_dir / "fonts")
    logo = resolve_logo(args.logo, asset_dir)
    year = season_start_year(boot)
    season = period["season"]
    title = "PREMIER LEAGUE GAME STATE"
    subtitle = f"Time losing / drawing / winning | {season} | {period['range']}"
    if period["provisional"]:
        subtitle += " | Provisional FPL scores"
    footer = f"{args.channel} | FPL scores + FotMob goals | {period['range']} | {dt.date.today():%d.%m.%Y}"
    note = f"All {len(period['fixtures'])} timelines verified. Regulation minutes only (90-minute cap; stoppage excluded)."
    import tempfile
    with tempfile.TemporaryDirectory(prefix="gamestate_", dir=str(root_dir)) as temp:
        temp_dir = Path(temp)
        outputs = _output_paths(temp_dir)
        texts = _text_paths(temp_dir)
        draw_game_state_chart(df, title, subtitle, footer, str(outputs["1"]),
                              logos, short, args.channel, logo, note, layout="1")
        _save_text(texts["1"], _game_state_post(
            df, gw, args.channel, "All 20 teams", period["season"], period["provisional"]))
        _save_report_metadata(temp_dir / "1.metadata.json", period, {
            "report": "GAME_STATE", "team_count": len(df),
            "verified_fixture_count": df.attrs["verified_fixture_count"],
            "matches_by_team": df.attrs["matches_by_team"],
            "time_basis": "regulation_90", "percentages_sum_to_100": True,
        })
        for generated in temp_dir.iterdir():
            generated.replace(out_dir / generated.name)
    for old_name in ("1A.png", "1B.png", "1A.txt", "1B.txt"):
        (out_dir / old_name).unlink(missing_ok=True)
    print(f"GAME STATE: full 20-team page updated through GW{gw}.")
    if in_colab():
        display(IPImage(str(out_dir / "1.png")))
    return _output_paths(out_dir)


game_state_outputs = game_state_main()
game_state_outputs


## 5 · Validate outputs
Each chart folder contains `1.png` and `1.txt` only.


In [ ]:
output_dir = resolve_out_dir(None)
for report_dir in (output_dir / 'xG', output_dir / 'DEFCON', output_dir / 'GAME_STATE'):
    print('\n' + str(report_dir))
    for artifact in sorted(report_dir.glob('*')):
        if artifact.suffix in ('.png', '.txt'):
            print(' -', artifact.name)
print('Social handles:', X_HANDLE, YOUTUBE_HANDLE)
